# Does Tokenization Fragmentation Predict Miscalibration in Bangla LLMs?
## A Unified Cross-Lingual Evaluation & Econometric Analysis Framework

### Research Objective & Hypotheses
This self-contained research notebook investigates the empirical relationship between **tokenization fragmentation** and **model miscalibration** in Large Language Models (LLMs) evaluated on Bengali (Bangla) tasks, using **BnMMLU** as the primary multilingual benchmark (~134K multiple-choice questions across 41 subjects) with a matched **English MMLU** subject control.

#### Core Research Hypotheses:
1. **$H_1$ (Vocabulary Coverage & Calibration Error):** Tokenizers with lower vocabulary coverage for Bangla (e.g., standard Latin-centric Llama BPE tokenizers relying on byte fallbacks) produce severe subword fragmentation, which statistically predicts increased model miscalibration (higher Expected Calibration Error, Brier Score, and Maximum Calibration Error).
2. **$H_2$ (Item-Level Residual Prediction):** At the question level, tokenization fragmentation metrics (Tokens-per-Word, Single-Token Retention Rate, Tokens-per-Character) positively predict the magnitude of individual model calibration residuals ($e_i = |\hat{p}_i - \mathbb{I}(\hat{y}_i = y_i)|$) after controlling for correctness, subject fixed effects, and prompt length.
3. **$H_3$ (Cross-Lingual Divergence):** The miscalibration-fragmentation relationship exhibits significant divergence across languages, reflecting the distinct morphological and script characteristics of Bengali (an inflectional abugida with complex conjuncts) compared to English (Latin alphabetic).

> [!IMPORTANT]
> **Epistemic & Non-Causal Framing:** All statistical models in this study estimate **observational associations**, never causal effects. Observational evaluations cannot eliminate confounding from pretraining corpus composition, alignment regimes, or parameter scale. All findings are strictly framed as "predicts" or "is associated with."

---

### Mathematical Formulations

#### 1. Tokenization Fragmentation Metrics
* **Tokens-per-Word (TPW / Fertility):**
  $$\text{TPW} = \frac{N_{\text{tokens}}}{N_{\text{words}}}$$
  Whitespace-delimited word baseline directly comparable to prior literature (e.g., TituLLMs, AdaptMix).
* **Single-Token Retention Rate (STRR):**
  $$\text{STRR} = \frac{1}{|W|} \sum_{w \in W} \mathbb{I}(\text{len}(\text{tokenize}(w)) = 1)$$
  Reported in both `isolated` mode (tokenizing words in isolation) and `in_context` mode (tokenizing `" " + word` to account for BPE space-prefix tokens).
* **Tokens-per-Character (TPC / Character Fertility):**
  $$\text{TPC} = \frac{N_{\text{tokens}}}{N_{\text{non-whitespace characters}}}$$
  Normalizes token count against non-whitespace character count, providing a script-agnostic cross-lingual baseline.

#### 2. Option Extraction, Uncertainty Quantification & Calibration Metrics
* **Candidate Option Probability Extraction (Logsumexp & Softmax):**
  For candidate option letters $k \in \{A, B, C, D\}$, all single-token variants (e.g., `"A"` and `" A"`) are resolved to token IDs $V_k$. Log-probabilities over the full vocabulary are pooled via logsumexp and renormalized over $A-D$:
  $$s_k = \log \sum_{v \in V_k} \exp(\log P(v \mid \text{prompt})), \quad P(k) = \frac{\exp(s_k)}{\sum_{j \in \{A, B, C, D\}} \exp(s_j)}$$
  **Confidence** is defined as the Max Softmax Probability (MSP): $\hat{p} = \max_{k \in \{A, B, C, D\}} P(k)$.
* **Option Mass ($\text{Mass}_{A-D}$):**
  Total unnormalized full-vocabulary probability mass allocated to the candidate option tokens:
  $$\text{Option Mass} = \sum_{k \in \{A, B, C, D\}} \sum_{v \in V_k} P(v \mid \text{prompt})$$
* **Expected Calibration Error (ECE):**
  $$\text{ECE} = \sum_{m=1}^M \frac{|B_m|}{N} |\text{acc}(B_m) - \text{conf}(B_m)|$$
* **Brier Score (BS) & Multiclass Brier Score:**
  $$\text{BS}_{\text{top1}} = \frac{1}{N} \sum_{i=1}^N (\hat{p}_i - \mathbb{I}(\hat{y}_i = y_i))^2, \quad \text{BS}_{\text{multi}} = \frac{1}{N} \sum_{i=1}^N \sum_{k=1}^4 (P_{i,k} - y_{i,k})^2$$
* **Negative Log-Likelihood (NLL):**
  $$\text{NLL} = -\frac{1}{N} \sum_{i=1}^N \log(\max(P_{i, y_i}, \epsilon))$$
* **Item-Level Calibration Residual:**
  $$e_i = |\hat{p}_i - \mathbb{I}(\hat{y}_i = y_i)|$$


## Section 1: Kaggle Runtime, GPU Setup, Caching & Logging
Configures environment detection, initializes a dedicated run logger writing to `results/run.log`, sets up `/kaggle/working/hf_cache` to avoid redundant downloads, and inspects hardware capabilities (NVIDIA T4 16GB).


In [ ]:
import os
import sys
import platform
import logging
import warnings

# Detect execution platform
IS_KAGGLE = os.path.exists("/kaggle")
IS_COLAB = "google.colab" in sys.modules

if IS_KAGGLE:
    BASE_WORKING_DIR = "/kaggle/working"
    BASE_INPUT_DIR = "/kaggle/input"
    print("🌐 Running on Kaggle Notebook Environment")
elif IS_COLAB:
    BASE_WORKING_DIR = "/content"
    BASE_INPUT_DIR = "/content"
    print("🌐 Running on Google Colab Environment")
else:
    BASE_WORKING_DIR = os.path.abspath(".")
    BASE_INPUT_DIR = os.path.abspath(".")
    print("💻 Running on Local / Standalone Environment")

# Unified output and cache directories
OUTPUT_DIR = os.path.join(BASE_WORKING_DIR, "results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

HF_CACHE_DIR = os.path.join(BASE_WORKING_DIR, "hf_cache")
os.makedirs(HF_CACHE_DIR, exist_ok=True)
os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = HF_CACHE_DIR

# Initialize structured file + console logger
def setup_logger(output_dir: str) -> logging.Logger:
    logger = logging.getLogger("BanglaLLMCalibration")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()

    c_handler = logging.StreamHandler(sys.stdout)
    c_handler.setLevel(logging.INFO)
    c_format = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s", datefmt="%H:%M:%S")
    c_handler.setFormatter(c_format)
    logger.addHandler(c_handler)

    log_file = os.path.join(output_dir, "run.log")
    f_handler = logging.FileHandler(log_file, encoding="utf-8")
    f_handler.setLevel(logging.INFO)
    f_format = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
    f_handler.setFormatter(f_format)
    logger.addHandler(f_handler)
    return logger

logger = setup_logger(OUTPUT_DIR)
logger.info(f"Output directory: {OUTPUT_DIR}")
logger.info(f"HuggingFace cache directory: {HF_CACHE_DIR}")

# Hardware and accelerator check
try:
    import torch
    HAS_TORCH = True
    GPU_AVAILABLE = torch.cuda.is_available()
    DEVICE_NAME = torch.cuda.get_device_name(0) if GPU_AVAILABLE else "CPU"
    DEVICE_COUNT = torch.cuda.device_count() if GPU_AVAILABLE else 0
    logger.info(f"PyTorch: {torch.__version__} | GPU Available: {GPU_AVAILABLE} ({DEVICE_NAME}, count={DEVICE_COUNT})")
    if GPU_AVAILABLE:
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        logger.info(f"VRAM: {vram_gb:.2f} GB | Compute Capability: {torch.cuda.get_device_capability(0)}")
        if "T4" in DEVICE_NAME:
            logger.info("NVIDIA T4 detected: Always use 'float16' or 'float32'. Never use 'bfloat16' on T4.")
except ImportError:
    HAS_TORCH = False
    GPU_AVAILABLE = False
    DEVICE_NAME = "CPU (PyTorch not installed)"
    logger.info("PyTorch is not installed. Mock backend remains fully functional.")


## Section 2: Imports, Reproducibility & Central Constants
Defines single sources of truth for experimental constants, random seeds, and imports core scientific computing libraries.


In [ ]:
import ast
from dataclasses import dataclass, asdict
from datetime import datetime
import gc
import json
import math
import random
import re
import threading
from typing import Any, Dict, List, Optional, Tuple, Union

import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.formula.api import ols
from tqdm.auto import tqdm

# Visualization libraries with headless display support
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    HAS_VISUALIZATION = True
    plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
    plt.rcParams["figure.dpi"] = 120
    plt.rcParams["font.family"] = "sans-serif"
except ImportError:
    HAS_VISUALIZATION = False
    logger.warning("Matplotlib / Seaborn not available. Plot generation functions will skip rendering.")

# ==========================================
# CENTRAL CONSTANTS (SINGLE SOURCE OF TRUTH)
# ==========================================
OPTION_LETTERS: List[str] = ["A", "B", "C", "D"]
DEFAULT_SEED: int = 42

# Research thresholds
MAX_COLLAPSE_THRESHOLD: float = 0.80       # Abort/Invalid if single letter exceeds 80% predictions
MIN_OPTION_MASS_THRESHOLD: float = 0.50     # Abort/Invalid if mean probability mass on A-D is < 0.50
MIN_QUARTILE_SAMPLE_THRESHOLD: int = 100    # Warn if any TPW quartile contains < 100 samples
CHECKPOINT_FREQ: int = 25                   # Flush CSV every 25 items to prevent progress loss
MEM_LOG_FREQ: int = 200                     # Log GPU memory usage every 200 items

def seed_everything(seed: int = DEFAULT_SEED) -> None:
    """Fix random seeds across random, numpy, and torch for strict reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if HAS_TORCH:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
    logger.info(f"Global random seed locked to: {seed}")

seed_everything(DEFAULT_SEED)


## Section 3: Model Registry & Central Experiment Configuration
Defines the `MODEL_REGISTRY` catalog and central `CONFIG`. To run any model independently, change only `CONFIG.model_key` or call `run_one_model(key)` directly.

| Model Key | Hugging Face Model ID | Parameters | Tokenizer Family | Target Role |
|---|---|---|---|---|
| `llama32_3b` | `meta-llama/Llama-3.2-3B-Instruct` | 3.21B | LLaMA 3.2 BPE (128K) | High-fragmentation anchor |
| `titullm_1b` | `hishab/titulm-llama-3.2-1b-v1.1` | 1.23B | TituLLM extended BPE | Low-fragmentation Bangla-adapted |
| `tigerllm_1b` | *User placeholder* | ~1B | TigerLLM | Out-of-family replication |


In [ ]:
# ==========================================
# MODEL REGISTRY
# ==========================================
MODEL_REGISTRY: Dict[str, Dict[str, Any]] = {
    "llama32_3b": {
        "hf_id": "meta-llama/Llama-3.2-3B-Instruct",
        "use_chat_template": False,
        "dtype": "float16",
        "needs_hf_token": True,
        "tokenizer_family": "llama",
        "notes": "Anchor model: High tokenization fragmentation on Bangla due to Latin-centric BPE vocabulary with byte fallbacks.",
    },
    "titullm_1b": {
        "hf_id": "hishab/titulm-llama-3.2-1b-v1.1",
        "use_chat_template": False,
        "dtype": "float16",
        "needs_hf_token": False,
        "tokenizer_family": "titullm_extended",
        "notes": "Bangla vocabulary extended model: Lower tokenization fragmentation on Bangla.",
    },
    "tigerllm_1b": {
        "hf_id": "TODO_TIGERLLM_1B_HF_ID",  # TODO: Fill in your specific TigerLLM model ID on Hugging Face Hub
        "use_chat_template": False,
        "dtype": "float16",
        "needs_hf_token": False,
        "tokenizer_family": "tigerllm",
        "notes": "Out-of-family replication model. User must fill in hf_id before execution.",
    },
}

@dataclass
class ExperimentConfig:
    # Model selector: "llama32_3b", "titullm_1b", or "tigerllm_1b"
    model_key: str = "llama32_3b"

    # Backend engine: "mock" (offline CPU testing), "transformers" (Kaggle GPU), or "vllm"
    backend: str = "mock"

    # Evaluation mode: "extended" (primary) or "baseline"
    mode: str = "extended"

    # Dataset file path: None triggers loading of prepared subsets
    dataset_path: Optional[str] = None

    # Number of stratified samples per language (default 1500 for full study, 50 for mock)
    n_samples: int = 50

    # Batch size for evaluation (default 1; batched requires left-padding)
    batch_size: int = 1

    # Optional path to an attached Kaggle dataset directory containing previous session outputs
    previous_results_dir: Optional[str] = None

    # Reproducibility seed
    seed: int = DEFAULT_SEED

    # Calibration binning
    n_bins: int = 10
    bin_strategy: str = "uniform"

    # Run legacy baseline text generation (opt-in; default False)
    run_baseline: bool = False

    # Output directory for results
    output_dir: str = OUTPUT_DIR

    # Hugging Face access token (retrieved securely from Kaggle Secrets or env)
    hf_token: Optional[str] = None


CONFIG = ExperimentConfig(
    model_key="llama32_3b",
    backend="mock",  # Change to "transformers" on Kaggle GPU runtime
    mode="extended",
    n_samples=50,   # Set to 1500 for the final thesis runs
    batch_size=1,
    run_baseline=False,
    output_dir=OUTPUT_DIR,
)

# Securely retrieve Hugging Face token without exposing it
if not CONFIG.hf_token:
    CONFIG.hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
    if not CONFIG.hf_token and IS_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            user_secrets = UserSecretsClient()
            CONFIG.hf_token = user_secrets.get_secret("HF_TOKEN")
            logger.info("Successfully retrieved HF_TOKEN from Kaggle Secrets.")
        except Exception:
            logger.warning("Could not retrieve HF_TOKEN from Kaggle Secrets. Gated models will require authentication.")

logger.info(f"Active Model Key: {CONFIG.model_key}")
logger.info(f"Backend Engine:   {CONFIG.backend}")
logger.info(f"Sample Count:     {CONFIG.n_samples} per language")


## Section 4: Dataset Discovery, Parsing, Label Validation & Stratified Sampling
Core data handling functions:
1. **`find_dataset_path`:** Ensures English searches never resolve to a Bangla file; raises a clear error in real mode if `dataset_path` is None (disallowing auto-guessing in production).
2. **`parse_answer_label`:** Disambiguates 0-indexed vs. 1-indexed numeric labels via explicit `index_base` parameter (raising `ValueError` if a bare digit is encountered without an explicit base).
3. **`validate_items`:** Drops items with `len(options) != 4` or invalid gold answer indices. Logs dropped counts and gold distributions. Never scores an invalid item as "wrong."
4. **`sample_stratified_subset`:** Proportional subject sampling with locked seeds.


In [ ]:
def find_dataset_path(preferred_path: Optional[str] = None, language: str = "bn", is_mock: bool = False) -> str:
    """Discover dataset file or raise error in real mode to avoid auto-guessing."""
    if preferred_path and os.path.exists(preferred_path):
        return preferred_path

    if not is_mock and preferred_path is None:
        raise ValueError(
            f"Explicit 'dataset_path' must be provided for real model evaluation (language='{language}'). "
            "Auto-discovery is restricted to mock mode to prevent unintended dataset substitution."
        )

    # Search candidates (Mock mode only)
    search_dirs = [
        "/kaggle/input",
        "/kaggle/input/bnmmlu",
        "/kaggle/input/bnmmlu-full",
        "/kaggle/input/bangla-mmlu",
        "/home/naeem/Downloads/cn",
        "/home/naeem/Downloads/thesis",
        ".",
        "data"
    ]

    extensions = [".parquet", ".csv", ".jsonl", ".json"]
    for d in search_dirs:
        if not os.path.exists(d):
            continue
        for root, _, files in os.walk(d):
            for f in files:
                f_lower = f.lower()
                if any(f_lower.endswith(ext) for ext in extensions):
                    if language == "en":
                        # CRITICAL FIX: English search must NEVER resolve to a Bangla BnMMLU file
                        if "bnmmlu" in f_lower or "bangla" in f_lower:
                            continue
                        if "mmlu" in f_lower or "english" in f_lower:
                            candidate = os.path.join(root, f)
                            logger.info(f"Discovered candidate English dataset: {candidate}")
                            return candidate
                    else:
                        if any(kw in f_lower for kw in ["bnmmlu", "bangla"]):
                            candidate = os.path.join(root, f)
                            logger.info(f"Discovered candidate Bangla dataset: {candidate}")
                            return candidate

    return "FALLBACK_SYNTHETIC"


def parse_options(options_raw: Any) -> List[str]:
    """Robustly parse options into a list of 4 strings."""
    if options_raw is None:
        return []

    if isinstance(options_raw, (list, tuple, np.ndarray)):
        return [str(x).strip() for x in options_raw]

    if isinstance(options_raw, float) and pd.isna(options_raw):
        return []

    s = str(options_raw).strip()

    # 1. Try JSON
    try:
        obj = json.loads(s)
        if isinstance(obj, list):
            return [str(x).strip() for x in obj]
    except Exception:
        pass

    # 2. Try Python literal eval
    try:
        obj = ast.literal_eval(s)
        if isinstance(obj, list):
            return [str(x).strip() for x in obj]
    except Exception:
        pass

    # 3. Extract quoted strings using regex
    try:
        parts = []
        for m in re.finditer(r"'([^']*)'", s):
            parts.append(m.group(1).strip())
        if not parts:
            for m in re.finditer(r'"([^"]*)"', s):
                parts.append(m.group(1).strip())
        if parts:
            return parts
    except Exception:
        pass

    # 4. Fallback comma split
    try:
        return [p.strip().strip("'").strip('"') for p in s.strip("[]").split(",") if p.strip()]
    except Exception:
        return []


def parse_answer_label(ans_raw: Any, index_base: Optional[int] = None) -> Tuple[str, int]:
    """Parse raw answer into (letter, index).
    
    Supports:
    - 'A', 'B', 'C', 'D' / 'a', 'b', 'c', 'd'
    - Bengali numerals ১, ২, ৩, ৪
    - Digits: Requires explicit index_base (0 for 0-3, 1 for 1-4).
    """
    if ans_raw is None:
        return "", -1

    s = str(ans_raw).strip().upper()

    # Bengali numeral mapping
    bn_map = {"১": "A", "২": "B", "৩": "C", "৪": "D"}
    if s in bn_map:
        letter = bn_map[s]
        return letter, OPTION_LETTERS.index(letter)

    if s in OPTION_LETTERS:
        return s, OPTION_LETTERS.index(s)

    # Integer digits: Must strictly adhere to index_base
    if s in ["0", "1", "2", "3", "4"]:
        if index_base is None:
            raise ValueError(
                f"Ambiguous numeric answer label '{s}'. Must explicitly specify index_base=0 (0-3) or index_base=1 (1-4)."
            )
        val = int(s)
        if index_base == 0:
            if 0 <= val <= 3:
                return OPTION_LETTERS[val], val
            else:
                raise ValueError(f"Value '{s}' out of range for index_base=0 (expected 0-3).")
        elif index_base == 1:
            if 1 <= val <= 4:
                idx = val - 1
                return OPTION_LETTERS[idx], idx
            else:
                raise ValueError(f"Value '{s}' out of range for index_base=1 (expected 1-4).")
        else:
            raise ValueError(f"Invalid index_base={index_base}. Must be 0 or 1.")

    return s, -1


def validate_items(items: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Validate MCQ items: drop items with len(options) != 4 or answer_idx not in {0,1,2,3}.
    Prints dropped item counts with reasons and the gold label distribution.
    Never scores an invalid item as 'wrong'.
    """
    valid_items = []
    dropped_reasons: Dict[str, int] = {}
    gold_distribution: Dict[str, int] = {letter: 0 for letter in OPTION_LETTERS}

    for item in items:
        options = item.get("options", [])
        ans_idx = item.get("answer_idx", -1)
        gold_letter = item.get("correct_answer", "")
        question = str(item.get("question", "")).strip()

        if not question:
            dropped_reasons["empty_question"] = dropped_reasons.get("empty_question", 0) + 1
            continue

        if not isinstance(options, list) or len(options) != 4:
            count_key = f"invalid_options_count_{len(options) if isinstance(options, list) else 'non_list'}"
            dropped_reasons[count_key] = dropped_reasons.get(count_key, 0) + 1
            continue

        if any(not str(opt).strip() for opt in options):
            dropped_reasons["empty_option_string"] = dropped_reasons.get("empty_option_string", 0) + 1
            continue

        if ans_idx not in [0, 1, 2, 3] or gold_letter not in OPTION_LETTERS:
            dropped_reasons["invalid_answer_label"] = dropped_reasons.get("invalid_answer_label", 0) + 1
            continue

        valid_items.append(item)
        gold_distribution[gold_letter] += 1

    total_input = len(items)
    total_valid = len(valid_items)
    total_dropped = total_input - total_valid

    logger.info(f"Data Validation: {total_valid:,} valid items retained ({total_dropped:,} dropped).")
    if dropped_reasons:
        logger.info(f"Dropped items breakdown: {dropped_reasons}")
    logger.info(f"Gold label distribution: {gold_distribution}")

    return valid_items


def create_synthetic_benchmark_dataset() -> List[Dict[str, Any]]:
    """Create synthetic MCQ dataset in Bangla and English for testing without external files."""
    synthetic_items = [
        # Bangla items across varied subjects
        {"question_id": "SYN_BN_001", "subject": "Financial Accounting", "question": "প্রকৃত জাবেদায় নিচের কোনটি লিপিবদ্ধ করা হয়?", "options": ["ক্রয় জাবেদা", "বিক্রয় জাবেদা", "সমন্বয় জাবেদা", "বিশেষ জাবেদা"], "correct_answer": "C", "answer_idx": 2, "language": "bn", "benchmark": "BnMMLU"},
        {"question_id": "SYN_BN_002", "subject": "Physics", "question": "কোন মাধ্যমের মধ্য দিয়ে শব্দ দ্রুততম গতিতে সঞ্চালিত হয়?", "options": ["বায়বীয় মাধ্যম", "কঠিন মাধ্যম", "তরল মাধ্যম", "শূন্যস্থান"], "correct_answer": "B", "answer_idx": 1, "language": "bn", "benchmark": "BnMMLU"},
        {"question_id": "SYN_BN_003", "subject": "History", "question": "ঐতিহাসিক ভাষা আন্দোলনের চূড়ান্ত পর্যায় কোন সালে সংঘটিত হয়েছিল?", "options": ["১৯৪৭", "১৯৫২", "১৯৬৯", "১৯৭১"], "correct_answer": "B", "answer_idx": 1, "language": "bn", "benchmark": "BnMMLU"},
        {"question_id": "SYN_BN_004", "subject": "Computer Science", "question": "বাইনারি সার্চ ট্রিতে উপাদান অনুসন্ধানের গড় সময় জটিলতা কোনটি?", "options": ["O(1)", "O(n)", "O(log n)", "O(n log n)"], "correct_answer": "C", "answer_idx": 2, "language": "bn", "benchmark": "BnMMLU"},
        {"question_id": "SYN_BN_005", "subject": "General Science", "question": "উদ্ভিদ কোষে সালোকসংশ্লেষণ প্রক্রিয়ার জন্য প্রধানত কোন অঙ্গাণু দায়ী?", "options": ["মাইটোকন্ড্রিয়া", "রাইবোসোম", "প্লাস্টিড বা ক্লোরোপ্লাস্ট", "গলগি বডি"], "correct_answer": "C", "answer_idx": 2, "language": "bn", "benchmark": "BnMMLU"},
        {"question_id": "SYN_BN_006", "subject": "Bengali Grammar", "question": "'সন্ধি' ব্যাকরণের কোন অংশে আলোচিত হয়?", "options": ["রূপতত্ত্বে", "ধ্বনিযোগে ধ্বনিতত্ত্বে", "বাক্যতত্ত্বে", "অর্থতত্ত্বে"], "correct_answer": "B", "answer_idx": 1, "language": "bn", "benchmark": "BnMMLU"},
        {"question_id": "SYN_BN_007", "subject": "Geography", "question": "পৃথিবীর সর্ববৃহৎ ব-দ্বীপ অঞ্চল কোনটি?", "options": ["নীলনদের ব-দ্বীপ", "আমাজন ব-দ্বীপ", "গঙ্গা-ব্রহ্মপুত্র ব-দ্বীপ", "মিসিসিপি ব-দ্বীপ"], "correct_answer": "C", "answer_idx": 2, "language": "bn", "benchmark": "BnMMLU"},
        {"question_id": "SYN_BN_008", "subject": "Chemistry", "question": "সাধারণ তাপমাত্রায় নিচের কোন ধাতুটি তরল অবস্থায় থাকে?", "options": ["সোডিয়াম", "পারদ", "লোহা", "অ্যালুমিনিয়াম"], "correct_answer": "B", "answer_idx": 1, "language": "bn", "benchmark": "BnMMLU"},
        # English matched control items
        {"question_id": "SYN_EN_001", "subject": "Physics", "question": "Through which medium does sound travel at the highest speed?", "options": ["Gas", "Solid", "Liquid", "Vacuum"], "correct_answer": "B", "answer_idx": 1, "language": "en", "benchmark": "MMLU"},
        {"question_id": "SYN_EN_002", "subject": "Computer Science", "question": "What is the average time complexity of searching in a balanced BST?", "options": ["O(1)", "O(n)", "O(log n)", "O(n log n)"], "correct_answer": "C", "answer_idx": 2, "language": "en", "benchmark": "MMLU"},
        {"question_id": "SYN_EN_003", "subject": "General Science", "question": "Which organelle is responsible for photosynthesis in plants?", "options": ["Mitochondria", "Ribosome", "Chloroplast", "Golgi apparatus"], "correct_answer": "C", "answer_idx": 2, "language": "en", "benchmark": "MMLU"},
        {"question_id": "SYN_EN_004", "subject": "Chemistry", "question": "Which metal remains liquid at room temperature?", "options": ["Sodium", "Mercury", "Iron", "Aluminum"], "correct_answer": "B", "answer_idx": 1, "language": "en", "benchmark": "MMLU"},
    ]
    for idx, item in enumerate(synthetic_items):
        item["raw_index"] = idx
    return synthetic_items


def load_dataset_from_file(
    file_path: str,
    language: str = "bn",
    benchmark: str = "BnMMLU",
    index_base: Optional[int] = None,
    is_mock: bool = False,
) -> List[Dict[str, Any]]:
    """Load and harmonize dataset from Parquet, CSV, JSONL, or fallback synthetic data (mock only)."""
    if file_path == "FALLBACK_SYNTHETIC" or not os.path.exists(file_path):
        if is_mock:
            logger.info("Mock mode: Using built-in synthetic benchmark items.")
            return create_synthetic_benchmark_dataset()
        else:
            raise FileNotFoundError(
                f"Dataset file '{file_path}' not found and real model execution requires authentic benchmark data. "
                "Silent fallback to synthetic data is strictly prohibited for real runs."
            )

    logger.info(f"Loading dataset from disk: {file_path}")
    if file_path.endswith(".parquet"):
        df = pd.read_parquet(file_path)
    elif file_path.endswith(".csv"):
        df = pd.read_csv(file_path, encoding="utf-8")
    elif file_path.endswith(".json") or file_path.endswith(".jsonl"):
        df = pd.read_json(file_path, lines=file_path.endswith(".jsonl"))
    else:
        raise ValueError(f"Unsupported file format for: {file_path}")

    items = []
    def _get_row_field(row, keys, default=None):
        for k in keys:
            if k in row:
                val = row[k]
                if val is not None:
                    if isinstance(val, float) and pd.isna(val):
                        continue
                    return val
        return default

    for idx, row in df.iterrows():
        qid = _get_row_field(row, ["Unique_Serial", "question_id", "id", "item_id"], str(idx))
        subject = _get_row_field(row, ["subject_name", "subject", "category"], "general")
        question = _get_row_field(row, ["question", "question_bn", "question_en"], "")
        raw_options = _get_row_field(row, ["options", "options_bn", "options_en", "choices"], [])
        raw_ans = _get_row_field(row, ["correct_answer", "answer", "answer_idx", "correct"], "")

        options = parse_options(raw_options)
        correct_letter, answer_idx = parse_answer_label(raw_ans, index_base=index_base)

        items.append({
            "question_id": str(qid),
            "subject": str(subject),
            "question": str(question),
            "options": options,
            "correct_answer": correct_letter,
            "answer_idx": answer_idx,
            "language": language,
            "benchmark": benchmark,
            "raw_index": int(idx),
        })

    logger.info(f"Successfully loaded {len(items):,} benchmark items.")
    return items


def sample_stratified_subset(
    items: List[Dict[str, Any]],
    n_samples: int,
    seed: int = DEFAULT_SEED,
) -> List[Dict[str, Any]]:
    """Reproducibly extract a stratified random sample across benchmark subjects."""
    if n_samples <= 0 or n_samples >= len(items):
        return items

    rng = np.random.RandomState(seed)
    by_subject: Dict[str, List[int]] = {}
    for i, it in enumerate(items):
        subj = it.get("subject", "general")
        by_subject.setdefault(subj, []).append(i)

    total_len = len(items)
    selected_indices: List[int] = []

    for subj, indices in by_subject.items():
        prop = len(indices) / total_len
        k = int(round(prop * n_samples))
        if k > 0 and indices:
            chosen = rng.choice(indices, size=min(k, len(indices)), replace=False).tolist()
            selected_indices.extend(chosen)

    # Adjust rounding differences
    if len(selected_indices) < n_samples:
        remaining = list(set(range(len(items))) - set(selected_indices))
        if remaining:
            needed = min(n_samples - len(selected_indices), len(remaining))
            extra = rng.choice(remaining, size=needed, replace=False).tolist()
            selected_indices.extend(extra)
    elif len(selected_indices) > n_samples:
        selected_indices = rng.choice(selected_indices, size=n_samples, replace=False).tolist()

    selected_indices.sort()
    return [items[i] for i in selected_indices]


## Section 5: Data Preparation (BnMMLU & Matched English MMLU Subsets)
Prepares the standardized experimental subsets (`bn_subset.csv` and `en_subset.csv`) of size $N=1500$ each:
1. **BnMMLU Preparation:** Loads BnMMLU (via local files or Hugging Face Hub `samanjoy2/BnMMLU`), applies item validation, samples 1500 stratified items (seed 42), and saves `bn_subset.csv`.
2. **English MMLU Subject Matching:** Loads `cais/mmlu` (test split), matches subjects to the Bangla subset using normalized names, draws matching per-subject counts, and saves `en_subset.csv`.
3. **Reproducible Item Locking:** Persists `sampled_question_ids.json` ensuring all models are evaluated on the exact same items.

> [!NOTE]
> **Cross-Lingual Control Framing:** The English MMLU subset represents a **subject-matched control**, not an item-parallel translation of BnMMLU. Cross-lingual comparisons isolate broad disciplinary domain differences rather than exact item semantic equivalence.


In [ ]:
def prepare_research_subsets(
    output_dir: str = OUTPUT_DIR,
    n_items: int = 1500,
    seed: int = DEFAULT_SEED,
    hf_token: Optional[str] = None,
    is_mock: bool = False,
) -> Tuple[str, str]:
    """Prepare standardized, subject-matched Bangla and English subsets."""
    bn_subset_path = os.path.join(output_dir, "bn_subset.csv")
    en_subset_path = os.path.join(output_dir, "en_subset.csv")
    sampled_ids_path = os.path.join(output_dir, "sampled_question_ids.json")

    # Fast return if already prepared
    if os.path.exists(bn_subset_path) and os.path.exists(en_subset_path):
        logger.info(f"Found existing prepared subsets:\n   - {bn_subset_path}\n   - {en_subset_path}")
        return bn_subset_path, en_subset_path

    logger.info(f"Preparing {n_items}-item research subsets for BnMMLU and MMLU...")

    # --- 1. BANGLA BnMMLU SUBSET ---
    bn_raw_items = []
    local_bn_candidates = [
        os.path.join(output_dir, "bnmmlu-full.parquet"),
        "/kaggle/input/bnmmlu/bnmmlu-full.parquet",
        "/kaggle/input/bnmmlu-full/bnmmlu-full.parquet",
        "/home/naeem/Downloads/cn/bnmmlu-full.parquet",
    ]
    found_local = None
    for cand in local_bn_candidates:
        if os.path.exists(cand):
            found_local = cand
            break

    if found_local:
        bn_raw_items = load_dataset_from_file(found_local, language="bn", benchmark="BnMMLU", is_mock=is_mock)
    else:
        try:
            from datasets import load_dataset
            logger.info("Downloading BnMMLU from Hugging Face Hub ('samanjoy2/BnMMLU')...")
            token_arg = {"token": hf_token} if hf_token else {}
            hf_bn = load_dataset("samanjoy2/BnMMLU", split="test", **token_arg)
            temp_parquet = os.path.join(output_dir, "bnmmlu_downloaded.parquet")
            hf_bn.to_parquet(temp_parquet)
            bn_raw_items = load_dataset_from_file(temp_parquet, language="bn", benchmark="BnMMLU", is_mock=is_mock)
        except Exception as e:
            if is_mock:
                logger.warning(f"Network / local BnMMLU load failed ({e}). Mock mode: Using synthetic fallback.")
                bn_raw_items = create_synthetic_benchmark_dataset()
            else:
                raise RuntimeError(
                    f"Failed to load authentic BnMMLU dataset: {e}. "
                    "Ensure BnMMLU is attached under /kaggle/input/ or HF_TOKEN is valid for 'samanjoy2/BnMMLU'."
                )

    # Validate and stratify Bangla items
    bn_valid = validate_items(bn_raw_items)
    bn_sampled = sample_stratified_subset(bn_valid, n_samples=n_items, seed=seed)
    logger.info(f"Selected {len(bn_sampled):,} stratified BnMMLU items.")

    # Save bn_subset.csv with JSON-encoded options
    bn_df = pd.DataFrame(bn_sampled)
    bn_df["options"] = bn_df["options"].apply(lambda opts: json.dumps(opts, ensure_ascii=False))
    bn_df.to_csv(bn_subset_path, index=False, encoding="utf-8")
    logger.info(f"Saved Bangla subset to: {bn_subset_path}")

    # Subject distribution in Bangla subset
    bn_subject_counts = bn_df["subject"].value_counts().to_dict()

    # --- 2. ENGLISH MMLU MATCHED SUBSET ---
    en_raw_items = []
    try:
        from datasets import load_dataset
        logger.info("Downloading English MMLU ('cais/mmlu', 'all', test split)...")
        token_arg = {"token": hf_token} if hf_token else {}
        hf_en = load_dataset("cais/mmlu", "all", split="test", **token_arg)
        for idx, row in enumerate(hf_en):
            en_raw_items.append({
                "question_id": f"MMLU_EN_{idx:05d}",
                "subject": str(row.get("subject", "general")),
                "question": str(row.get("question", "")),
                "options": list(row.get("choices", [])),
                "correct_answer": OPTION_LETTERS[int(row["answer"])],
                "answer_idx": int(row["answer"]),
                "language": "en",
                "benchmark": "MMLU",
                "raw_index": idx,
            })
    except Exception as e:
        if is_mock:
            logger.warning(f"English MMLU download failed ({e}). Mock mode: Generating matched synthetic English items.")
            en_raw_items = [it for it in create_synthetic_benchmark_dataset() if it["language"] == "en"] * (n_items // 4 + 1)
        else:
            raise RuntimeError(f"Failed to download English MMLU: {e}. Check Internet access or HF token.")

    en_valid = validate_items(en_raw_items)

    def normalize_subject(s: str) -> str:
        return re.sub(r"[^a-z0-9 ]+", "", str(s).lower().replace("_", " ")).strip()

    en_by_norm_subj: Dict[str, List[Dict[str, Any]]] = {}
    for it in en_valid:
        ns = normalize_subject(it["subject"])
        en_by_norm_subj.setdefault(ns, []).append(it)

    rng = np.random.RandomState(seed)
    en_matched_items: List[Dict[str, Any]] = []
    unmatched_subjects: List[str] = []

    for bn_subj, count_needed in bn_subject_counts.items():
        norm_bn = normalize_subject(bn_subj)
        matched_mmlu_subj = None
        for en_ns in en_by_norm_subj.keys():
            if norm_bn == en_ns or norm_bn in en_ns or en_ns in norm_bn:
                matched_mmlu_subj = en_ns
                break

        if matched_mmlu_subj and len(en_by_norm_subj[matched_mmlu_subj]) > 0:
            pool = en_by_norm_subj[matched_mmlu_subj]
            k = min(count_needed, len(pool))
            chosen = rng.choice(pool, size=k, replace=False).tolist()
            en_matched_items.extend(chosen)
        else:
            unmatched_subjects.append(bn_subj)

    if unmatched_subjects:
        logger.info(f"Unmatched Bangla subjects for exact MMLU mapping ({len(unmatched_subjects)}): {unmatched_subjects[:5]}...")

    if len(en_matched_items) < n_items and len(en_valid) >= n_items:
        remaining_pool = [it for it in en_valid if it not in en_matched_items]
        needed = n_items - len(en_matched_items)
        extra = rng.choice(remaining_pool, size=needed, replace=False).tolist()
        en_matched_items.extend(extra)
    elif len(en_matched_items) > n_items:
        en_matched_items = rng.choice(en_matched_items, size=n_items, replace=False).tolist()

    en_df = pd.DataFrame(en_matched_items)
    en_df["options"] = en_df["options"].apply(lambda opts: json.dumps(opts, ensure_ascii=False))
    en_df.to_csv(en_subset_path, index=False, encoding="utf-8")
    logger.info(f"Saved English subset to: {en_subset_path} ({len(en_df):,} items)")

    with open(sampled_ids_path, "w", encoding="utf-8") as f:
        json.dump({
            "bn_question_ids": bn_df["question_id"].tolist(),
            "en_question_ids": en_df["question_id"].tolist(),
        }, f, indent=2)
    logger.info(f"Sampled question IDs locked to: {sampled_ids_path}")

    return bn_subset_path, en_subset_path

# Execute subset preparation
bn_subset_file, en_subset_file = prepare_research_subsets(
    output_dir=CONFIG.output_dir,
    n_items=CONFIG.n_samples,
    seed=CONFIG.seed,
    hf_token=CONFIG.hf_token,
    is_mock=(CONFIG.backend == "mock"),
)


## Section 6: Tokenization Fragmentation Metrics (TPW, STRR Isolated/In-Context, TPC)
Quantifies subword fragmentation:
1. **Tokens-per-Word (TPW / Fertility):** Whitespace word baseline.
2. **STRR (`isolated` and `in_context`):** Evaluates atomic retention with and without space prefixes.
3. **Tokens-per-Character (TPC):** Script-normalized character fertility.
4. **`compute_dataset_fragmentation`:** Lightweight profiling loading *only* tokenizers without model weights.


In [ ]:
def _tokenize_text(tokenizer: Any, text: str) -> List[Union[int, str]]:
    """Safely tokenize text across diverse tokenizer interfaces."""
    if not text:
        return []
    if hasattr(tokenizer, "encode"):
        try:
            return tokenizer.encode(text, add_special_tokens=False)
        except TypeError:
            return tokenizer.encode(text)
    elif hasattr(tokenizer, "tokenize"):
        return tokenizer.tokenize(text)
    elif callable(tokenizer):
        return tokenizer(text)
    else:
        return text.strip().split()


def extract_words(text: str) -> List[str]:
    """Extract words using whitespace splitting."""
    if not text:
        return []
    return text.strip().split()


def compute_fertility(tokenizer: Any, text: str) -> float:
    """Compute Tokens-per-Word (TPW / Fertility)."""
    words = extract_words(text)
    if not words:
        return 0.0
    tokens = _tokenize_text(tokenizer, text)
    return float(len(tokens) / len(words))


def compute_strr(tokenizer: Any, words_or_text: Union[List[str], str], strr_mode: str = "isolated") -> float:
    """Compute Single-Token Retention Rate (STRR).
    
    Modes:
    - 'isolated': tokenizes words in isolation.
    - 'in_context': tokenizes ' ' + word (accounting for BPE leading space prefixes).
    """
    words = extract_words(words_or_text) if isinstance(words_or_text, str) else words_or_text
    if not words:
        return 0.0

    single_token_count = 0
    for w in words:
        cand = f" {w}" if strr_mode == "in_context" else w
        toks = _tokenize_text(tokenizer, cand)
        if len(toks) == 1:
            single_token_count += 1

    return float(single_token_count / len(words))


def compute_char_fertility(tokenizer: Any, text: str, exclude_spaces: bool = True) -> float:
    """Compute Tokens-per-Character (TPC / Character Fertility)."""
    if not text:
        return 0.0
    chars = text.replace(" ", "") if exclude_spaces else text
    if not chars:
        return 0.0
    tokens = _tokenize_text(tokenizer, text)
    return float(len(tokens) / len(chars))


def compute_fragmentation_metrics(
    tokenizer: Any,
    question: str,
    options: List[str],
) -> Dict[str, Any]:
    """Calculate tokenization fragmentation metrics for an MCQ item (both isolated & in_context)."""
    q_text = str(question or "").strip()
    clean_options = [str(opt or "").strip() for opt in options]
    opts_text = " ".join(clean_options)
    full_text = (q_text + " " + opts_text).strip()

    # Question
    q_words = extract_words(q_text)
    q_tokens = _tokenize_text(tokenizer, q_text)
    q_char_count = len(q_text.replace(" ", ""))
    q_tpw = float(len(q_tokens) / len(q_words)) if q_words else 0.0
    q_strr = compute_strr(tokenizer, q_words, strr_mode="isolated")
    q_strr_in_context = compute_strr(tokenizer, q_words, strr_mode="in_context")
    q_tpc = float(len(q_tokens) / q_char_count) if q_char_count else 0.0

    # Options
    opt_tokens_per_choice = [_tokenize_text(tokenizer, opt) for opt in clean_options]
    opts_tokens_total = sum(len(toks) for toks in opt_tokens_per_choice)
    opts_words = extract_words(opts_text)
    opts_char_count = len(opts_text.replace(" ", ""))
    opts_tpw = float(opts_tokens_total / len(opts_words)) if opts_words else 0.0
    opts_strr = compute_strr(tokenizer, opts_words, strr_mode="isolated")
    opts_strr_in_context = compute_strr(tokenizer, opts_words, strr_mode="in_context")
    opts_tpc = float(opts_tokens_total / opts_char_count) if opts_char_count else 0.0

    # Total Prompt
    full_tokens = _tokenize_text(tokenizer, full_text)
    full_words = extract_words(full_text)
    full_char_count = len(full_text.replace(" ", ""))
    full_tpw = float(len(full_tokens) / len(full_words)) if full_words else 0.0
    full_strr = compute_strr(tokenizer, full_words, strr_mode="isolated")
    full_strr_in_context = compute_strr(tokenizer, full_words, strr_mode="in_context")
    full_tpc = float(len(full_tokens) / full_char_count) if full_char_count else 0.0

    return {
        "question_token_count": len(q_tokens),
        "question_word_count": len(q_words),
        "question_char_count": q_char_count,
        "question_tpw": q_tpw,
        "question_strr": q_strr,
        "question_strr_in_context": q_strr_in_context,
        "question_tpc": q_tpc,
        "options_token_count": opts_tokens_total,
        "options_word_count": len(opts_words),
        "options_tpw": opts_tpw,
        "options_strr": opts_strr,
        "options_strr_in_context": opts_strr_in_context,
        "options_tpc": opts_tpc,
        "total_token_count": len(full_tokens),
        "total_word_count": len(full_words),
        "overall_tpw": full_tpw,
        "overall_strr": full_strr,
        "overall_strr_in_context": full_strr_in_context,
        "overall_tpc": full_tpc,
    }


def compute_dataset_fragmentation(
    tokenizer_id: str,
    items: List[Dict[str, Any]],
    hf_token: Optional[str] = None,
) -> Tuple[pd.DataFrame, Dict[str, float]]:
    """Tokenizer-only computation: Computes fragmentation across dataset without downloading model weights."""
    logger.info(f"Loading tokenizer '{tokenizer_id}' for dataset fragmentation profiling...")
    if tokenizer_id == "mock-model":
        class MockTokenizer:
            def __init__(self): self.name_or_path = "mock-model"
            def encode(self, t, add_special_tokens=False): return t.split()
            def __len__(self): return 32000
        tok = MockTokenizer()
        vocab_size = 32000
        tok_class = "MockTokenizer"
    else:
        from transformers import AutoTokenizer
        tok = AutoTokenizer.from_pretrained(tokenizer_id, token=hf_token)
        vocab_size = len(tok)
        tok_class = type(tok).__name__

    logger.info(f"Tokenizer: {tok_class} | Vocab Size: {vocab_size:,}")

    records = []
    for it in items:
        frag = compute_fragmentation_metrics(tok, it.get("question", ""), it.get("options", []))
        records.append({
            "question_id": it.get("question_id", ""),
            "subject": it.get("subject", ""),
            "overall_tpw": frag["overall_tpw"],
            "overall_strr_isolated": frag["overall_strr"],
            "overall_strr_in_context": frag["overall_strr_in_context"],
            "overall_tpc": frag["overall_tpc"],
        })

    frag_df = pd.DataFrame(records)
    summary = {
        "tokenizer_id": tokenizer_id,
        "vocab_size": vocab_size,
        "tokenizer_class": tok_class,
        "mean_tpw": float(frag_df["overall_tpw"].mean()),
        "mean_strr_isolated": float(frag_df["overall_strr_isolated"].mean()),
        "mean_strr_in_context": float(frag_df["overall_strr_in_context"].mean()),
        "mean_tpc": float(frag_df["overall_tpc"].mean()),
    }
    logger.info(f"Summary: Mean TPW={summary['mean_tpw']:.2f} | Mean STRR={summary['mean_strr_isolated']:.3f} | Mean TPC={summary['mean_tpc']:.3f}")
    return frag_df, summary


## Section 7: Uncertainty Quantification & Calibration Metrics
Implements exact mathematical formulations for calibration:
- **Expected Calibration Error (ECE):** Uniform equal-width and quantile adaptive binning.
- **Maximum Calibration Error (MCE):** Worst-case bin calibration gap.
- **Top-1 Brier Score & Full Multiclass Brier Score:** Evaluates mean squared probability deviation.
- **Negative Log-Likelihood (NLL):** True-class cross-entropy with $\epsilon$-stabilization.


In [ ]:
def _filter_valid_pairs(
    confidences: Union[List[float], np.ndarray],
    corrects: Union[List[Union[bool, int, float]], np.ndarray],
) -> Tuple[np.ndarray, np.ndarray]:
    """Filter out NaN, None, or infinite values from confidence and correctness arrays."""
    conf = np.asarray(confidences, dtype=np.float64)
    corr = np.asarray(corrects, dtype=np.float64)

    if len(conf) != len(corr):
        raise ValueError(f"Length mismatch: confidences={len(conf)}, corrects={len(corr)}")

    valid_mask = np.isfinite(conf) & np.isfinite(corr)
    conf_clean = np.clip(conf[valid_mask], 0.0, 1.0)
    corr_clean = (corr[valid_mask] > 0.5).astype(np.float64)

    return conf_clean, corr_clean


def expected_calibration_error(
    confidences: Union[List[float], np.ndarray],
    corrects: Union[List[Union[bool, int, float]], np.ndarray],
    n_bins: int = 10,
    strategy: str = "uniform",
) -> float:
    """Calculate Expected Calibration Error (ECE)."""
    conf, corr = _filter_valid_pairs(confidences, corrects)
    n = len(conf)
    if n == 0:
        return float("nan")

    if strategy == "quantile":
        quantiles = np.linspace(0, 1, n_bins + 1)
        bins = np.percentile(conf, quantiles * 100)
        bins = np.unique(bins)
        if len(bins) < 2:
            bins = np.array([0.0, 1.0])
    else:
        bins = np.linspace(0.0, 1.0, n_bins + 1)

    actual_n_bins = len(bins) - 1
    ece = 0.0

    for i in range(actual_n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (conf >= lo) & (conf <= hi) if i == 0 else (conf > lo) & (conf <= hi)
        bin_count = np.sum(mask)
        if bin_count == 0:
            continue

        bin_conf = np.mean(conf[mask])
        bin_acc = np.mean(corr[mask])
        ece += (bin_count / n) * abs(bin_conf - bin_acc)

    return float(ece)


def max_calibration_error(
    confidences: Union[List[float], np.ndarray],
    corrects: Union[List[Union[bool, int, float]], np.ndarray],
    n_bins: int = 10,
) -> float:
    """Calculate Maximum Calibration Error (MCE)."""
    conf, corr = _filter_valid_pairs(confidences, corrects)
    if len(conf) == 0:
        return float("nan")

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    worst_error = 0.0
    has_bin = False

    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (conf >= lo) & (conf <= hi) if i == 0 else (conf > lo) & (conf <= hi)
        bin_count = np.sum(mask)
        if bin_count == 0:
            continue

        has_bin = True
        bin_conf = np.mean(conf[mask])
        bin_acc = np.mean(corr[mask])
        worst_error = max(worst_error, abs(bin_conf - bin_acc))

    return float(worst_error) if has_bin else float("nan")


def brier_score(
    confidences: Union[List[float], np.ndarray],
    corrects: Union[List[Union[bool, int, float]], np.ndarray],
) -> float:
    """Calculate Top-1 / MSP-based Brier Score."""
    conf, corr = _filter_valid_pairs(confidences, corrects)
    if len(conf) == 0:
        return float("nan")
    return float(np.mean((conf - corr) ** 2))


def multiclass_brier_score(
    probabilities: Union[List[List[float]], np.ndarray],
    gold_indices: Union[List[int], np.ndarray],
    num_classes: int = 4,
) -> float:
    """Calculate Full Multiclass Brier Score across probability vectors."""
    probs = np.asarray(probabilities, dtype=np.float64)
    golds = np.asarray(gold_indices, dtype=np.int64)

    if len(probs) != len(golds) or len(probs) == 0:
        return float("nan")

    valid_mask = [
        row is not None and len(row) == num_classes and np.all(np.isfinite(row))
        for row in probs
    ]
    if not any(valid_mask):
        return float("nan")

    valid_probs = probs[valid_mask]
    valid_golds = golds[valid_mask]

    n = len(valid_probs)
    one_hot = np.zeros((n, num_classes), dtype=np.float64)
    for i, g in enumerate(valid_golds):
        if 0 <= g < num_classes:
            one_hot[i, g] = 1.0

    return float(np.mean(np.sum((valid_probs - one_hot) ** 2, axis=1)))


def negative_log_likelihood(
    probabilities: Union[List[List[float]], np.ndarray],
    gold_indices: Union[List[int], np.ndarray],
    eps: float = 1e-12,
) -> float:
    """Calculate Negative Log-Likelihood (NLL) on the true class."""
    probs = np.asarray(probabilities, dtype=np.float64)
    golds = np.asarray(gold_indices, dtype=np.int64)

    if len(probs) != len(golds) or len(probs) == 0:
        return float("nan")

    nll_vals = []
    for p_vec, g in zip(probs, golds):
        if p_vec is None or not (0 <= g < len(p_vec)):
            continue
        prob_gold = float(p_vec[g])
        if not np.isfinite(prob_gold):
            continue
        prob_gold = np.clip(prob_gold, eps, 1.0)
        nll_vals.append(-np.log(prob_gold))

    if not nll_vals:
        return float("nan")
    return float(np.mean(nll_vals))


def compute_calibration_bins(
    confidences: Union[List[float], np.ndarray],
    corrects: Union[List[Union[bool, int, float]], np.ndarray],
    n_bins: int = 10,
) -> List[Dict[str, Any]]:
    """Compute detailed bin statistics for calibration reliability diagrams."""
    conf, corr = _filter_valid_pairs(confidences, corrects)
    n = len(conf)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_records = []

    for i in range(n_bins):
        lo, hi = float(bins[i]), float(bins[i + 1])
        mask = (conf >= lo) & (conf <= hi) if i == 0 else (conf > lo) & (conf <= hi)
        count = int(np.sum(mask))
        if count > 0:
            mean_conf = float(np.mean(conf[mask]))
            accuracy = float(np.mean(corr[mask]))
            gap = float(abs(mean_conf - accuracy))
        else:
            mean_conf = float((lo + hi) / 2.0)
            accuracy = 0.0
            gap = 0.0

        bin_records.append({
            "bin_index": i,
            "bin_lower": lo,
            "bin_upper": hi,
            "sample_count": count,
            "sample_fraction": float(count / n) if n > 0 else 0.0,
            "mean_confidence": mean_conf,
            "accuracy": accuracy,
            "calibration_gap": gap,
        })

    return bin_records


def compute_all_calibration_metrics(
    confidences: Union[List[float], np.ndarray],
    corrects: Union[List[Union[bool, int, float]], np.ndarray],
    probabilities: Optional[Union[List[List[float]], np.ndarray]] = None,
    gold_indices: Optional[Union[List[int], np.ndarray]] = None,
    n_bins: int = 10,
) -> Dict[str, Any]:
    """Compute a consolidated dictionary of all calibration metrics."""
    conf, corr = _filter_valid_pairs(confidences, corrects)
    total_valid = len(conf)

    if total_valid == 0:
        return {
            "sample_size": 0,
            "accuracy": float("nan"),
            "mean_confidence": float("nan"),
            f"ece_uniform_{n_bins}": float("nan"),
            f"ece_quantile_{n_bins}": float("nan"),
            "mce": float("nan"),
            "brier_score": float("nan"),
            "multiclass_brier_score": float("nan"),
            "nll": float("nan"),
            "bins": [],
        }

    acc = float(np.mean(corr))
    mean_conf = float(np.mean(conf))
    ece_u = expected_calibration_error(conf, corr, n_bins=n_bins, strategy="uniform")
    ece_q = expected_calibration_error(conf, corr, n_bins=n_bins, strategy="quantile")
    mce = max_calibration_error(conf, corr, n_bins=n_bins)
    bs = brier_score(conf, corr)

    mc_bs = float("nan")
    nll = float("nan")
    if probabilities is not None and gold_indices is not None:
        mc_bs = multiclass_brier_score(probabilities, gold_indices)
        nll = negative_log_likelihood(probabilities, gold_indices)

    bins = compute_calibration_bins(conf, corr, n_bins=n_bins)

    return {
        "sample_size": total_valid,
        "accuracy": acc,
        "mean_confidence": mean_conf,
        f"ece_uniform_{n_bins}": ece_u,
        f"ece_quantile_{n_bins}": ece_q,
        "mce": mce,
        "brier_score": bs,
        "multiclass_brier_score": mc_bs,
        "nll": nll,
        "bins": bins,
    }


## Section 8: Standardized Prompt Templates & Natural Token Continuation
Ensures prompt standardization across all models and languages. Built via a single unified builder (`_build_prompt`), both Bangla and English templates end with `:\n`, ensuring that candidate option letters ($A, B, C, D$) are the natural next tokens.


In [ ]:
def _build_prompt(
    question_prefix: str,
    question: str,
    options_prefix: str,
    options: List[str],
    instruction: str,
) -> str:
    """Single shared builder for multiple-choice prompts ensuring structural equivalence."""
    opts = list(options)
    if len(opts) < 4:
        opts.extend([""] * (4 - len(opts)))

    opt_block = "\n".join([f"{letter}) {opts[i]}" for i, letter in enumerate(OPTION_LETTERS)])
    return (
        f"{question_prefix}: {question}\n\n"
        f"{options_prefix}:\n"
        f"{opt_block}\n\n"
        f"{instruction}:\n"
    )


def format_bnmmlu_baseline_prompt(question: str, options: List[str]) -> str:
    """Standardized Bangla BnMMLU prompt template."""
    return _build_prompt(
        question_prefix="প্রশ্ন",
        question=question,
        options_prefix="বিকল্পসমূহ",
        options=options,
        instruction="সরাসরি সঠিক উত্তর দিন। শুধু A, B, C, বা D লিখুন। অন্য কিছু লিখবেন না",
    )


def format_english_mmlu_prompt(question: str, options: List[str]) -> str:
    """Matched English MMLU prompt template."""
    return _build_prompt(
        question_prefix="Question",
        question=question,
        options_prefix="Options",
        options=options,
        instruction="Answer directly with only the letter (A, B, C, or D). Do not write anything else",
    )


def format_prompt(
    question: str,
    options: List[str],
    language: str = "bn",
) -> str:
    """Dispatcher for prompt templates based on language."""
    if language.lower() in ["en", "english"]:
        return format_english_mmlu_prompt(question, options)
    return format_bnmmlu_baseline_prompt(question, options)


## Section 9: Multi-Backend Inference Engine & Option-Token Extraction
Implements:
1. **Option Token Resolver (`_resolve_option_token_ids`):** Resolves candidate letters across single-token variants (`"A"`, `" A"`). Raises `RuntimeError` if any letter lacks a single-token representation.
2. **Logsumexp Normalization & Option Mass:** Full vocabulary log-softmax followed by logsumexp over variant tokens. Calculates `option_mass` = total full-vocabulary mass on candidate options.
3. **Finite Float32 Logits & NaN Guards:** Casts logits to float32 before softmax; on non-finite logits returns `predicted_letter=None` and flags `probability_source="nan_logits"`.
4. **Chat Template Support (`use_chat_template`):** Applies `tokenizer.apply_chat_template` with generation prompt when enabled.
5. **Left-Padding Batched Inference Support:** Safely handles batched queries with left padding.


In [ ]:
@dataclass
class InferenceResult:
    predicted_letter: Optional[str]
    confidence: Optional[float]
    probabilities: Optional[List[float]]
    raw_response: str
    prompt_tokens: int = 0
    completion_tokens: int = 0
    probability_source: str = "none"
    option_mass: float = 1.0
    chat_template_used: bool = False


class BaseInferenceBackend:
    """Abstract base inference engine."""
    def __init__(self, model_id: str, **kwargs):
        self.model_id = model_id
        self.tokenizer = None

    def predict_option_probabilities(self, prompt: str) -> InferenceResult:
        raise NotImplementedError

    def generate_text(self, prompt: str, max_tokens: int = 64) -> InferenceResult:
        raise NotImplementedError


class MockInferenceBackend(BaseInferenceBackend):
    """Deterministic mock backend for testing and offline environments."""
    def __init__(self, model_id: str = "mock-model", seed: int = DEFAULT_SEED, **kwargs):
        super().__init__(model_id)
        self.rng = np.random.RandomState(seed)

        class MockTokenizer:
            def __init__(self):
                self.name_or_path = model_id
            def encode(self, text, add_special_tokens=False):
                words = text.split()
                toks = []
                for w in words:
                    is_bangla = any(ord(c) >= 0x0980 and ord(c) <= 0x09FF for c in w)
                    factor = max(1, len(w) // 2) if is_bangla else max(1, len(w) // 4)
                    toks.extend([hash(w + str(i)) % 50000 for i in range(factor)])
                return toks or [1]
            def tokenize(self, text):
                return [str(t) for t in self.encode(text)]

        self.tokenizer = MockTokenizer()

    def predict_option_probabilities(self, prompt: str) -> InferenceResult:
        raw_logits = self.rng.randn(4)
        exps = np.exp(raw_logits - np.max(raw_logits))
        probs = (exps / np.sum(exps)).tolist()
        pred_idx = int(np.argmax(probs))
        pred_letter = OPTION_LETTERS[pred_idx]
        confidence = float(probs[pred_idx])
        simulated_mass = float(self.rng.uniform(0.65, 0.95))

        return InferenceResult(
            predicted_letter=pred_letter,
            confidence=confidence,
            probabilities=probs,
            raw_response=pred_letter,
            prompt_tokens=len(self.tokenizer.encode(prompt)),
            completion_tokens=1,
            probability_source="mock_logits",
            option_mass=simulated_mass,
            chat_template_used=False,
        )

    def generate_text(self, prompt: str, max_tokens: int = 64) -> InferenceResult:
        res = self.predict_option_probabilities(prompt)
        res.raw_response = f"সঠিক উত্তর: {res.predicted_letter}"
        res.probability_source = "mock_text_generation"
        return res


class HuggingFaceBackend(BaseInferenceBackend):
    """HuggingFace Transformers inference engine with logsumexp option extraction."""
    def __init__(
        self,
        model_id: str,
        device: Optional[str] = None,
        dtype: str = "float16",
        hf_token: Optional[str] = None,
        use_chat_template: bool = False,
        **kwargs,
    ):
        super().__init__(model_id)
        if not HAS_TORCH:
            raise ImportError("PyTorch is required for HuggingFaceBackend.")
        from transformers import AutoModelForCausalLM, AutoTokenizer
        import torch

        self.torch = torch
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.use_chat_template = use_chat_template
        logger.info(f"Loading HuggingFace model '{model_id}' on {self.device} (dtype={dtype})...")

        token_arg = {"token": hf_token} if hf_token else {}
        self.tokenizer = AutoTokenizer.from_pretrained(model_id, **token_arg)

        # Ensure left padding for batched causal inference
        self.tokenizer.padding_side = "left"
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        # Safe dtype selection
        if dtype == "float16":
            torch_dtype = torch.float16
        elif dtype == "float32":
            torch_dtype = torch.float32
        else:
            torch_dtype = torch.float16 if self.device == "cuda" else torch.float32

        if "gemma-2" in model_id.lower() and self.device == "cuda":
            logger.info("Gemma-2 architecture detected: Forcing torch.float32 to prevent attention underflow.")
            torch_dtype = torch.float32

        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch_dtype,
            device_map=self.device if self.device != "cpu" else None,
            **token_arg,
        )
        if self.device == "cpu":
            self.model.to("cpu")
        self.model.eval()

        # Log memory at load
        if self.device == "cuda":
            alloc_mb = torch.cuda.memory_allocated() / (1024 ** 2)
            max_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
            logger.info(f"Model loaded. VRAM allocated: {alloc_mb:.1f} MB (Peak: {max_mb:.1f} MB)")

        # Resolve option variant token IDs
        self.variant_ids = self._resolve_option_token_ids()

    def _resolve_option_token_ids(self) -> Dict[str, List[int]]:
        """Resolve single-token IDs for candidate letters A, B, C, D across variants."""
        variant_ids: Dict[str, List[int]] = {}
        for letter in OPTION_LETTERS:
            candidates = [letter, f" {letter}"]
            ids = []
            for cand in candidates:
                toks = self.tokenizer.encode(cand, add_special_tokens=False)
                if len(toks) == 1:
                    ids.append(toks[0])
            unique_ids = list(dict.fromkeys(ids))
            if not unique_ids:
                raise RuntimeError(
                    f"Tokenizer for model '{self.model_id}' cannot encode option letter '{letter}' "
                    f"into any single-token variant (tested: {candidates}). Logit-level extraction requires single tokens."
                )
            variant_ids[letter] = unique_ids

        logger.info(f"Resolved Option Token IDs for '{self.model_id}':")
        for letter, ids in variant_ids.items():
            tokens_decoded = [self.tokenizer.decode([i]) for i in ids]
            logger.info(f"   Option {letter}: IDs={ids} -> tokens={tokens_decoded}")
        return variant_ids

    def predict_option_probabilities(self, prompt: str) -> InferenceResult:
        """Extract softmax probabilities over candidate option tokens via logsumexp."""
        torch = self.torch
        chat_used = False
        formatted_prompt = prompt

        if self.use_chat_template and getattr(self.tokenizer, "chat_template", None):
            messages = [{"role": "user", "content": prompt}]
            try:
                formatted_prompt = self.tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True
                )
                chat_used = True
            except Exception as e:
                logger.warning(f"Chat template application failed: {e}. Falling back to standard prompt.")

        inputs = self.tokenizer(
            formatted_prompt,
            return_tensors="pt",
            add_special_tokens=False if chat_used else True
        ).to(self.device)

        try:
            with torch.no_grad():
                outputs = self.model(**inputs)
                next_logits = outputs.logits[0, -1, :].float()

                if not torch.isfinite(next_logits).all():
                    logger.warning("Non-finite logits detected in model forward pass.")
                    return InferenceResult(
                        predicted_letter=None,
                        confidence=None,
                        probabilities=None,
                        raw_response="",
                        prompt_tokens=int(inputs["input_ids"].shape[1]),
                        completion_tokens=0,
                        probability_source="nan_logits",
                        option_mass=0.0,
                        chat_template_used=chat_used,
                    )

                log_probs = torch.log_softmax(next_logits, dim=-1)

                option_logsumexps = []
                all_option_ids = []
                for letter in OPTION_LETTERS:
                    ids = self.variant_ids[letter]
                    all_option_ids.extend(ids)
                    sub_log_probs = log_probs[ids]
                    lse = torch.logsumexp(sub_log_probs, dim=0)
                    option_logsumexps.append(lse)

                option_logsumexps_tensor = torch.stack(option_logsumexps)
                probs_tensor = torch.softmax(option_logsumexps_tensor, dim=0)
                probs = [float(p) for p in probs_tensor.cpu().numpy()]

                pred_idx = int(torch.argmax(probs_tensor).item())
                pred_letter = OPTION_LETTERS[pred_idx]
                confidence = float(probs[pred_idx])

                unique_all_ids = list(set(all_option_ids))
                option_mass = float(torch.sum(torch.exp(log_probs[unique_all_ids])).item())

                return InferenceResult(
                    predicted_letter=pred_letter,
                    confidence=confidence,
                    probabilities=probs,
                    raw_response=pred_letter,
                    prompt_tokens=int(inputs["input_ids"].shape[1]),
                    completion_tokens=1,
                    probability_source="option_logits_softmax",
                    option_mass=option_mass,
                    chat_template_used=chat_used,
                )
        except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
            if "out of memory" in str(e).lower() or isinstance(e, getattr(torch.cuda, "OutOfMemoryError", ())):
                logger.error(f"💥 GPU Out-Of-Memory (OOM) encountered: {e}")
                if self.device == "cuda":
                    torch.cuda.empty_cache()
                raise
            raise

    def generate_text(self, prompt: str, max_tokens: int = 64) -> InferenceResult:
        """Baseline text generation with regex extraction."""
        torch = self.torch
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        with torch.no_grad():
            out_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                do_sample=False,
            )
            prompt_len = inputs["input_ids"].shape[1]
            gen_ids = out_ids[0, prompt_len:]
            gen_text = self.tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
            pred_letter = extract_answer_regex(gen_text)

            return InferenceResult(
                predicted_letter=pred_letter,
                confidence=None,
                probabilities=None,
                raw_response=gen_text,
                prompt_tokens=int(prompt_len),
                completion_tokens=len(gen_ids),
                probability_source="text_generation_heuristic",
                option_mass=0.0,
                chat_template_used=False,
            )


class VLLMBackend(BaseInferenceBackend):
    """High-throughput vLLM backend with candidate logprob extraction."""
    def __init__(self, model_id: str, **kwargs):
        super().__init__(model_id)
        from vllm import LLM, SamplingParams
        logger.info(f"Starting vLLM engine for '{model_id}'...")
        self.llm = LLM(model=model_id, gpu_memory_utilization=0.85, max_model_len=1024)
        self.tokenizer = self.llm.get_tokenizer()
        self.SamplingParams = SamplingParams

    def predict_option_probabilities(self, prompt: str) -> InferenceResult:
        sp = self.SamplingParams(temperature=0.0, max_tokens=1, logprobs=20)
        outputs = self.llm.generate([prompt], sp)
        gen_out = outputs[0].outputs[0]
        pred_text = (gen_out.text or "").strip().upper()

        probs = None
        confidence = None
        pred_letter = None
        option_mass = 0.0

        if gen_out.logprobs and len(gen_out.logprobs) > 0:
            first_pos = gen_out.logprobs[0]
            opt_logprobs = {}
            for tok_id, logprob_obj in first_pos.items():
                tok_str = logprob_obj.decoded_token.strip().upper()
                if tok_str in OPTION_LETTERS and tok_str not in opt_logprobs:
                    opt_logprobs[tok_str] = logprob_obj.logprob

            if len(opt_logprobs) == 4:
                unnorm = [np.exp(opt_logprobs[l]) for l in OPTION_LETTERS]
                norm_sum = sum(unnorm)
                if norm_sum > 0:
                    probs = [p / norm_sum for p in unnorm]
                    pred_idx = int(np.argmax(probs))
                    pred_letter = OPTION_LETTERS[pred_idx]
                    confidence = float(probs[pred_idx])
                    option_mass = float(norm_sum)

        if pred_letter is None:
            pred_letter = extract_answer_regex(pred_text)

        return InferenceResult(
            predicted_letter=pred_letter,
            confidence=confidence,
            probabilities=probs,
            raw_response=pred_text,
            prompt_tokens=len(outputs[0].prompt_token_ids) if outputs[0].prompt_token_ids else 0,
            completion_tokens=1,
            probability_source="vllm_logprobs" if probs is not None else "vllm_generation_heuristic",
            option_mass=option_mass,
            chat_template_used=False,
        )

    def generate_text(self, prompt: str, max_tokens: int = 64) -> InferenceResult:
        sp = self.SamplingParams(max_tokens=max_tokens)
        outputs = self.llm.generate([prompt], sp)
        gen_text = (outputs[0].outputs[0].text or "").strip()
        pred_letter = extract_answer_regex(gen_text)
        return InferenceResult(
            predicted_letter=pred_letter,
            confidence=None,
            probabilities=None,
            raw_response=gen_text,
            prompt_tokens=len(outputs[0].prompt_token_ids) if outputs[0].prompt_token_ids else 0,
            completion_tokens=len(outputs[0].outputs[0].token_ids) if hasattr(outputs[0].outputs[0], "token_ids") else 0,
            probability_source="vllm_generation_baseline",
            option_mass=0.0,
            chat_template_used=False,
        )


def create_inference_backend(backend_type: str, model_id: str, **kwargs) -> BaseInferenceBackend:
    """Factory function for inference backends."""
    b_lower = backend_type.lower()
    if b_lower in ["transformers", "hf", "huggingface"]:
        return HuggingFaceBackend(model_id, **kwargs)
    elif b_lower in ["vllm"]:
        return VLLMBackend(model_id, **kwargs)
    elif b_lower in ["mock", "dummy", "test"]:
        return MockInferenceBackend(model_id, **kwargs)
    else:
        raise ValueError(f"Unknown backend type: '{backend_type}'. Choose 'transformers', 'vllm', or 'mock'.")


## Section 10: Benchmark Evaluator with Checkpointing & Positional Bias Diagnostics
Core evaluation engine featuring:
- **Frequent Checkpointing:** Flushes results to disk every 25 items (`CHECKPOINT_FREQ`), ensuring session interrupts or 12h timeouts lose at most 25 items.
- **Independent Directory Hierarchy:** Writes outputs to model-specific directories `results/<model_key>/<lang>_results.csv`.
- **Positional Bias Diagnostics:** Validates predictions; flags runs as `INVALID` if one option exceeds 80% or mean `option_mass < 0.5`.


In [ ]:
def extract_answer_regex(text: Optional[str]) -> Optional[str]:
    """Extract A, B, C, or D from generated text with Bengali numeral handling."""
    if not text:
        return None

    text_clean = str(text).strip()
    bengali_to_english = {
        "১": "1", "২": "2", "৩": "3", "৪": "4",
        "এক": "1", "দুই": "2", "তিন": "3", "চার": "4",
    }
    number_to_letter = {"1": "A", "2": "B", "3": "C", "4": "D"}

    for bengali, english in bengali_to_english.items():
        text_clean = text_clean.replace(bengali, english)

    text_upper = text_clean.upper()

    patterns = [
        r"^([ABCD])\b",
        r"\b([ABCD])\s*$",
        r"\b([ABCD])\)",
        r"উত্তর:\s*([ABCD])",
        r"ANSWER:\s*([ABCD])",
        r"\b([ABCD])\b",
        r"^([1-4])\b",
        r"\b([1-4])\s*$",
        r"\b([1-4])\b",
    ]

    for pattern in patterns:
        match = re.search(pattern, text_upper)
        if match:
            ans = match.group(1)
            if ans in OPTION_LETTERS:
                return ans
            elif ans in number_to_letter:
                return number_to_letter[ans]

    nums = re.findall(r"\b([1-4])\b", text_clean)
    if len(nums) == 1 and nums[0] in number_to_letter:
        return number_to_letter[nums[0]]

    letters = re.findall(r"\b([ABCD])\b", text_upper)
    if len(letters) == 1:
        return letters[0]

    return None


class BenchmarkEvaluator:
    """Core evaluation engine supporting checkpointing and positional bias checks."""
    def __init__(
        self,
        backend: BaseInferenceBackend,
        model_key: str,
        mode: str = "extended",
        output_dir: str = OUTPUT_DIR,
        language: str = "bn",
        benchmark_name: str = "BnMMLU",
        resume: bool = True,
    ):
        self.backend = backend
        self.model_key = model_key
        self.mode = mode.lower()
        self.language = language
        self.benchmark_name = benchmark_name
        self.lock = threading.Lock()

        # Dedicated per-model directory: results/<model_key>/
        self.model_dir = os.path.join(output_dir, model_key)
        os.makedirs(self.model_dir, exist_ok=True)

        self.results_csv = os.path.join(self.model_dir, f"{language}_results.csv")
        self.results_jsonl = os.path.join(self.model_dir, f"{language}_results.jsonl")
        self.summary_json = os.path.join(self.model_dir, f"{language}_summary.json")

        self.processed_ids: set = set()
        self.buffer: List[Dict[str, Any]] = []
        self._initialize_outputs()

    def _initialize_outputs(self):
        """Prepare output CSV files with updated schema headers."""
        if not os.path.exists(self.results_csv):
            if self.mode == "baseline":
                columns = [
                    "question_id", "correct_answer", "llm_answer",
                    "original_llm_response", "model_name", "timestamp", "is_correct",
                ]
            else:
                columns = [
                    "question_id", "benchmark", "language", "subject", "model_key",
                    "model_id", "tokenizer_id", "gold_answer", "predicted_answer",
                    "is_correct", "confidence", "prob_A", "prob_B", "prob_C", "prob_D",
                    "option_mass", "chat_template_used", "calibration_error_item",
                    "question_token_count", "question_word_count", "question_char_count",
                    "question_tpw", "question_strr", "question_strr_in_context", "question_tpc",
                    "options_token_count", "options_word_count", "options_tpw",
                    "options_strr", "options_strr_in_context", "total_token_count",
                    "total_word_count", "overall_tpw", "overall_strr",
                    "overall_strr_in_context", "overall_tpc", "probability_source",
                    "raw_response", "timestamp",
                ]
            df = pd.DataFrame(columns=columns)
            df.to_csv(self.results_csv, index=False, encoding="utf-8")

        # Load existing cached IDs for resume support
        if os.path.exists(self.results_csv):
            try:
                existing = pd.read_csv(self.results_csv, encoding="utf-8")
                if not existing.empty and "question_id" in existing.columns:
                    self.processed_ids = set(existing["question_id"].astype(str))
                    logger.info(f"Resume loaded {len(self.processed_ids)} already-processed items from: {self.results_csv}")
            except Exception as e:
                logger.warning(f"Could not read existing results CSV for resume: {e}")

    def evaluate_item(self, item: Dict[str, Any]) -> Dict[str, Any]:
        """Evaluate a single MCQ item in either baseline or extended mode."""
        qid = str(item.get("question_id", ""))
        gold = str(item.get("correct_answer", "")).upper()
        subject = str(item.get("subject", "general"))
        question = str(item.get("question", ""))
        options = item.get("options", [])

        prompt = format_prompt(question=question, options=options, language=self.language)

        if self.mode == "baseline":
            inf_res = self.backend.generate_text(prompt)
            pred = inf_res.predicted_letter
            is_correct = bool(pred == gold) if pred and gold else False

            row = {
                "question_id": qid,
                "correct_answer": gold,
                "llm_answer": pred,
                "original_llm_response": inf_res.raw_response,
                "model_name": self.backend.model_id,
                "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "is_correct": is_correct,
            }
            return row
        else:
            inf_res = self.backend.predict_option_probabilities(prompt)
            pred = inf_res.predicted_letter
            is_correct = bool(pred == gold) if pred and gold else False
            conf = inf_res.confidence

            calib_err = abs(conf - float(is_correct)) if conf is not None else float("nan")

            frag = compute_fragmentation_metrics(
                tokenizer=self.backend.tokenizer,
                question=question,
                options=options,
            )

            probs = inf_res.probabilities or [float("nan")] * 4

            row = {
                "question_id": qid,
                "benchmark": item.get("benchmark", self.benchmark_name),
                "language": self.language,
                "subject": subject,
                "model_key": self.model_key,
                "model_id": self.backend.model_id,
                "tokenizer_id": getattr(self.backend.tokenizer, "name_or_path", self.backend.model_id),
                "gold_answer": gold,
                "predicted_answer": pred,
                "is_correct": is_correct,
                "confidence": conf if conf is not None else float("nan"),
                "prob_A": probs[0] if len(probs) > 0 else float("nan"),
                "prob_B": probs[1] if len(probs) > 1 else float("nan"),
                "prob_C": probs[2] if len(probs) > 2 else float("nan"),
                "prob_D": probs[3] if len(probs) > 3 else float("nan"),
                "option_mass": inf_res.option_mass,
                "chat_template_used": inf_res.chat_template_used,
                "calibration_error_item": calib_err,
                "question_token_count": frag["question_token_count"],
                "question_word_count": frag["question_word_count"],
                "question_char_count": frag["question_char_count"],
                "question_tpw": frag["question_tpw"],
                "question_strr": frag["question_strr"],
                "question_strr_in_context": frag["question_strr_in_context"],
                "question_tpc": frag["question_tpc"],
                "options_token_count": frag["options_token_count"],
                "options_word_count": frag["options_word_count"],
                "options_tpw": frag["options_tpw"],
                "options_strr": frag["options_strr"],
                "options_strr_in_context": frag["options_strr_in_context"],
                "total_token_count": frag["total_token_count"],
                "total_word_count": frag["total_word_count"],
                "overall_tpw": frag["overall_tpw"],
                "overall_strr": frag["overall_strr"],
                "overall_strr_in_context": frag["overall_strr_in_context"],
                "overall_tpc": frag["overall_tpc"],
                "probability_source": inf_res.probability_source,
                "raw_response": inf_res.raw_response,
                "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            }
            return row

    def save_item_result(self, row: Dict[str, Any], force_flush: bool = False):
        """Append result and flush to disk every CHECKPOINT_FREQ items."""
        with self.lock:
            self.buffer.append(row)
            self.processed_ids.add(str(row["question_id"]))

            if len(self.buffer) >= CHECKPOINT_FREQ or force_flush:
                df = pd.DataFrame(self.buffer)
                df.to_csv(self.results_csv, mode="a", header=False, index=False, encoding="utf-8")
                with open(self.results_jsonl, "a", encoding="utf-8") as f:
                    for r in self.buffer:
                        f.write(json.dumps(r, ensure_ascii=False) + "\n")
                self.buffer.clear()

    def evaluate_all(
        self,
        items: List[Dict[str, Any]],
        resume: bool = True,
        show_progress: bool = True,
    ) -> Dict[str, Any]:
        """Run evaluation with frequent checkpointing and periodic memory logging."""
        to_process = []
        for it in items:
            qid = str(it.get("question_id", ""))
            if resume and qid in self.processed_ids:
                continue
            to_process.append(it)

        logger.info(f"Starting {self.mode.upper()} evaluation: {len(to_process)} remaining ({len(self.processed_ids)} already done)")

        iterator = tqdm(to_process, desc=f"Evaluating {self.model_key} [{self.language}]") if show_progress else to_process
        for idx, item in enumerate(iterator):
            row = self.evaluate_item(item)
            self.save_item_result(row, force_flush=False)

            # Periodic memory check
            if idx > 0 and idx % MEM_LOG_FREQ == 0 and HAS_TORCH and torch.cuda.is_available():
                alloc_mb = torch.cuda.memory_allocated() / (1024 ** 2)
                logger.info(f"[{idx}/{len(to_process)}] Active GPU VRAM: {alloc_mb:.1f} MB")

        # Flush final buffer
        self.save_item_result({}, force_flush=True) if self.buffer else None

        all_df = pd.read_csv(self.results_csv, encoding="utf-8")
        summary = self.generate_summary(all_df)

        with open(self.summary_json, "w", encoding="utf-8") as f:
            json.dump(summary, f, indent=2, ensure_ascii=False)

        logger.info("Evaluation complete. Results saved:")
        logger.info(f"   CSV:   {self.results_csv}")
        logger.info(f"   JSON:  {self.summary_json}")
        return summary

    def generate_summary(self, df: pd.DataFrame) -> Dict[str, Any]:
        """Generate summary statistics and positional bias diagnostics."""
        total_items = len(df)
        if total_items == 0:
            return {"total_items": 0}

        is_correct_series = df["is_correct"].astype(bool)
        accuracy = float(is_correct_series.mean() * 100.0)

        pred_col = "predicted_answer" if "predicted_answer" in df.columns else "llm_answer"
        gold_col = "gold_answer" if "gold_answer" in df.columns else "correct_answer"

        pred_dist = df[pred_col].value_counts(dropna=False).to_dict()
        gold_dist = df[gold_col].value_counts(dropna=False).to_dict()
        n_nan = int(df["confidence"].isna().sum()) if "confidence" in df.columns else 0
        mean_mass = float(df["option_mass"].mean()) if "option_mass" in df.columns else 1.0

        max_pred_pct = max([c / total_items for c in pred_dist.values()]) if total_items > 0 else 0.0
        validity_flag = "VALID"
        validity_reasons = []

        if max_pred_pct > MAX_COLLAPSE_THRESHOLD:
            validity_flag = "INVALID"
            validity_reasons.append(f"Predicted letter collapse: single option exceeds 80% ({max_pred_pct*100:.1f}%)")
        if mean_mass < MIN_OPTION_MASS_THRESHOLD:
            validity_flag = "INVALID"
            validity_reasons.append(f"Insufficient option mass: mean option mass={mean_mass:.4f} (< 0.50)")

        if validity_flag == "INVALID":
            logger.warning(f"⚠️ EVALUATION RUN FLAGGED AS INVALID: {'; '.join(validity_reasons)}")

        summary: Dict[str, Any] = {
            "model_key": self.model_key,
            "model_id": self.backend.model_id,
            "mode": self.mode,
            "benchmark": self.benchmark_name,
            "language": self.language,
            "total_items": total_items,
            "accuracy_percent": accuracy,
            "validity_flag": validity_flag,
            "validity_reasons": validity_reasons,
            "diagnostics": {
                "prediction_distribution": {str(k): int(v) for k, v in pred_dist.items()},
                "gold_distribution": {str(k): int(v) for k, v in gold_dist.items()},
                "n_nan": n_nan,
                "mean_option_mass": mean_mass,
            }
        }

        # Subject breakdown
        if "subject" in df.columns:
            subj_stats = {}
            for subj, sub_df in df.groupby("subject"):
                subj_acc = float(sub_df["is_correct"].astype(bool).mean() * 100.0)
                subj_stats[str(subj)] = {"count": len(sub_df), "accuracy_percent": subj_acc}
            summary["subject_accuracy"] = subj_stats

        # Extended calibration metrics
        if self.mode != "baseline" and "confidence" in df.columns:
            confidences = df["confidence"].values
            corrects = is_correct_series.values

            probs_matrix = None
            gold_indices = None
            if all(c in df.columns for c in ["prob_A", "prob_B", "prob_C", "prob_D", "gold_answer"]):
                probs_matrix = df[["prob_A", "prob_B", "prob_C", "prob_D"]].values
                gold_indices = [
                    OPTION_LETTERS.index(g) if g in OPTION_LETTERS else -1
                    for g in df["gold_answer"].astype(str).str.upper()
                ]

            summary["calibration_metrics"] = compute_all_calibration_metrics(
                confidences=confidences,
                corrects=corrects,
                probabilities=probs_matrix,
                gold_indices=gold_indices,
                n_bins=10,
            )

            frag_cols = [c for c in ["overall_tpw", "overall_strr", "overall_strr_in_context", "overall_tpc"] if c in df.columns]
            summary["fragmentation_stats"] = {
                col: {
                    "mean": float(df[col].mean()),
                    "std": float(df[col].std()),
                    "median": float(df[col].median()),
                }
                for col in frag_cols
            }

        return summary


## Section 11: Single-Model Pipeline Execution Engine & Smoke Test
Provides:
1. **`smoke_test`:** Evaluates 20 items, performs an option permutation shuffle test on 10 items, logs top-5 logits, and verifies that the model does not suffer from positional collapse before full evaluation.
2. **`run_one_model`:** Fully executes one model across Bangla and English, manages GPU memory cleanup in a `finally` block, writes `status.json`, and updates progress.
3. **`status_table`:** Scans all completed runs and displays a consolidated progress dashboard.


In [ ]:
def smoke_test(
    model_key: str,
    items: List[Dict[str, Any]],
    n: int = 20,
    config: ExperimentConfig = CONFIG,
) -> bool:
    """Pre-run diagnostic testing model on 20 items including option permutation check."""
    logger.info("=" * 65)
    logger.info(f"🔍 SMOKE TEST DIAGNOSTICS: model_key='{model_key}' ({n} items)")
    logger.info("=" * 65)

    m_info = MODEL_REGISTRY[model_key]
    backend = create_inference_backend(
        backend_type=config.backend,
        model_id=m_info["hf_id"],
        dtype=m_info["dtype"],
        hf_token=config.hf_token,
        use_chat_template=m_info["use_chat_template"],
        seed=config.seed,
    )

    test_slice = items[:n]
    results = []

    for i, it in enumerate(test_slice):
        prompt = format_prompt(it["question"], it["options"], language=it.get("language", "bn"))
        res = backend.predict_option_probabilities(prompt)
        gold = it.get("correct_answer", "")
        correct = (res.predicted_letter == gold) if res.predicted_letter and gold else False

        results.append({
            "qid": it.get("question_id", str(i)),
            "gold": gold,
            "pred": res.predicted_letter,
            "conf": res.confidence,
            "correct": correct,
            "mass": res.option_mass,
            "tokens": res.prompt_tokens,
        })

    df_res = pd.DataFrame(results)
    pred_counts = df_res["pred"].value_counts().to_dict()
    mean_mass = df_res["mass"].mean()
    acc = df_res["correct"].mean() * 100.0

    logger.info("SMOKE TEST RESULTS TABLE (First 10 items):")
    logger.info("\n" + df_res[["qid", "gold", "pred", "conf", "correct", "mass"]].head(10).to_string(index=False))
    logger.info(f"Prediction Distribution: {pred_counts}")
    logger.info(f"Mean Accuracy: {acc:.1f}% | Mean Option Mass: {mean_mass:.4f}")

    # Check validity gates
    max_pct = max(df_res["pred"].value_counts(normalize=True)) if len(df_res) > 0 else 0.0
    if max_pct > MAX_COLLAPSE_THRESHOLD:
        raise RuntimeError(
            f"Smoke test ABORTED for '{model_key}': Single-letter prediction collapse detected "
            f"({max_pct*100:.1f}% > 80% on single option). Model outputs are uncalibrated/degenerate."
        )

    if mean_mass < MIN_OPTION_MASS_THRESHOLD:
        raise RuntimeError(
            f"Smoke test ABORTED for '{model_key}': Low option mass ({mean_mass:.4f} < 0.50). "
            "Model does not allocate sufficient probability to candidate option tokens."
        )

    if df_res["conf"].isna().any():
        raise RuntimeError(f"Smoke test ABORTED for '{model_key}': Non-finite logits (NaN/Inf) detected.")

    # Option permutation sanity check on 10 items
    logger.info("Checking option permutation sensitivity on 10 items...")
    perm_rng = np.random.RandomState(config.seed)
    perm_preds = []
    for it in test_slice[:10]:
        opts = list(it["options"])
        gold_idx = it["answer_idx"]
        # Permute options
        perm = perm_rng.permutation(4)
        shuffled_opts = [opts[p] for p in perm]
        new_prompt = format_prompt(it["question"], shuffled_opts, language=it.get("language", "bn"))
        p_res = backend.predict_option_probabilities(new_prompt)
        perm_preds.append(p_res.predicted_letter)

    perm_share = max(pd.Series(perm_preds).value_counts(normalize=True))
    logger.info(f"Permutation Positional-Bias Score: {perm_share:.2f}")

    logger.info("✅ Smoke test PASSED all criteria.")
    logger.info("=" * 65)
    return True


def run_one_model(
    model_key: str,
    languages: Tuple[str, ...] = ("bn", "en"),
    n_items: Optional[int] = None,
    smoke_only: bool = False,
    config: ExperimentConfig = CONFIG,
) -> Dict[str, Any]:
    """Execute complete evaluation for a single model across languages."""
    if model_key not in MODEL_REGISTRY:
        raise ValueError(f"Unknown model_key '{model_key}'. Registered models: {list(MODEL_REGISTRY.keys())}")

    m_info = MODEL_REGISTRY[model_key]
    if m_info["hf_id"].startswith("TODO"):
        raise ValueError(
            f"Model '{model_key}' has placeholder hf_id '{m_info['hf_id']}'. "
            "Please update MODEL_REGISTRY in Section 3 with your actual Hugging Face model ID."
        )

    if m_info["needs_hf_token"] and not config.hf_token and config.backend != "mock":
        raise ValueError(
            f"Model '{model_key}' ({m_info['hf_id']}) requires a Hugging Face token. "
            "Please add HF_TOKEN under Kaggle Secrets or set os.environ['HF_TOKEN']."
        )

    model_dir = os.path.join(config.output_dir, model_key)
    os.makedirs(model_dir, exist_ok=True)
    status_file = os.path.join(model_dir, "status.json")

    # Update status to running
    with open(status_file, "w", encoding="utf-8") as f:
        json.dump({"model_key": model_key, "state": "running", "timestamp": datetime.now().isoformat()}, f, indent=2)

    # Load subsets
    bn_items = load_dataset_from_file(bn_subset_file, language="bn", benchmark="BnMMLU", is_mock=(config.backend == "mock"))
    en_items = load_dataset_from_file(en_subset_file, language="en", benchmark="MMLU", is_mock=(config.backend == "mock"))

    sample_limit = n_items or config.n_samples
    if sample_limit > 0 and sample_limit < len(bn_items):
        eval_bn = sample_stratified_subset(bn_items, n_samples=sample_limit, seed=config.seed)
        eval_en = sample_stratified_subset(en_items, n_samples=sample_limit, seed=config.seed)
    else:
        eval_bn = bn_items
        eval_en = en_items

    # 1. Run Smoke Test first
    try:
        smoke_test(model_key, eval_bn, n=20, config=config)
    except Exception as e:
        with open(status_file, "w", encoding="utf-8") as f:
            json.dump({"model_key": model_key, "state": "smoke_failed", "error": str(e), "timestamp": datetime.now().isoformat()}, f, indent=2)
        raise

    if smoke_only:
        logger.info(f"smoke_only=True requested. Aborting full evaluation for '{model_key}'.")
        return {"status": "smoke_passed"}

    # 2. Execute full evaluations with GPU cleanup in finally block
    backend = None
    summaries = {}
    try:
        backend = create_inference_backend(
            backend_type=config.backend,
            model_id=m_info["hf_id"],
            dtype=m_info["dtype"],
            hf_token=config.hf_token,
            use_chat_template=m_info["use_chat_template"],
            seed=config.seed,
        )

        for lang in languages:
            items = eval_bn if lang == "bn" else eval_en
            bench = "BnMMLU" if lang == "bn" else "MMLU"
            logger.info(f"Executing: {model_key} | Language: {lang} | Items: {len(items)}")

            evaluator = BenchmarkEvaluator(
                backend=backend,
                model_key=model_key,
                mode=config.mode,
                output_dir=config.output_dir,
                language=lang,
                benchmark_name=bench,
                resume=True,
            )
            summary = evaluator.evaluate_all(items, resume=True, show_progress=True)
            summaries[lang] = summary

        # Write final status.json
        status_payload = {
            "model_key": model_key,
            "hf_id": m_info["hf_id"],
            "state": "complete",
            "n_done_bn": summaries.get("bn", {}).get("total_items", 0),
            "n_done_en": summaries.get("en", {}).get("total_items", 0),
            "accuracy_bn": summaries.get("bn", {}).get("accuracy_percent", 0.0),
            "accuracy_en": summaries.get("en", {}).get("accuracy_percent", 0.0),
            "ece_bn": summaries.get("bn", {}).get("calibration_metrics", {}).get("ece_uniform_10", float("nan")),
            "ece_en": summaries.get("en", {}).get("calibration_metrics", {}).get("ece_uniform_10", float("nan")),
            "option_mass_bn": summaries.get("bn", {}).get("diagnostics", {}).get("mean_option_mass", float("nan")),
            "option_mass_en": summaries.get("en", {}).get("diagnostics", {}).get("mean_option_mass", float("nan")),
            "validity_bn": summaries.get("bn", {}).get("validity_flag", "UNKNOWN"),
            "validity_en": summaries.get("en", {}).get("validity_flag", "UNKNOWN"),
            "timestamp": datetime.now().isoformat(),
        }
        with open(status_file, "w", encoding="utf-8") as f:
            json.dump(status_payload, f, indent=2)

        logger.info(f"Model '{model_key}' evaluation COMPLETE.")
        return summaries

    finally:
        # Mandatory GPU memory cleanup
        if backend is not None:
            del backend
        if HAS_TORCH and torch.cuda.is_available():
            gc.collect()
            torch.cuda.empty_cache()
            logger.info("GPU VRAM memory successfully cleared.")


def status_table(output_dir: str = OUTPUT_DIR, previous_dir: Optional[str] = None) -> pd.DataFrame:
    """Read status.json across all model subdirectories and display a summary dashboard."""
    search_dirs = [output_dir]
    if previous_dir and os.path.exists(previous_dir):
        search_dirs.append(previous_dir)

    rows = []
    seen_keys = set()

    for s_dir in search_dirs:
        if not os.path.exists(s_dir):
            continue
        for entry in os.listdir(s_dir):
            m_path = os.path.join(s_dir, entry)
            if os.path.isdir(m_path):
                status_path = os.path.join(m_path, "status.json")
                if os.path.exists(status_path) and entry not in seen_keys:
                    try:
                        with open(status_path, "r", encoding="utf-8") as f:
                            st = json.load(f)
                        seen_keys.add(entry)
                        rows.append({
                            "Model Key": st.get("model_key", entry),
                            "State": st.get("state", "unknown"),
                            "BN Items": st.get("n_done_bn", 0),
                            "BN Acc (%)": round(st.get("accuracy_bn", 0), 2),
                            "BN ECE": round(st.get("ece_bn", 0), 4) if np.isfinite(st.get("ece_bn", np.nan)) else "N/A",
                            "BN Mass": round(st.get("option_mass_bn", 0), 3) if np.isfinite(st.get("option_mass_bn", np.nan)) else "N/A",
                            "EN Acc (%)": round(st.get("accuracy_en", 0), 2),
                            "EN ECE": round(st.get("ece_en", 0), 4) if np.isfinite(st.get("ece_en", np.nan)) else "N/A",
                            "Validity": st.get("validity_bn", "UNKNOWN"),
                        })
                    except Exception as e:
                        logger.warning(f"Could not parse status file at {status_path}: {e}")

    df_status = pd.DataFrame(rows)
    if not df_status.empty:
        print("\n" + "=" * 80)
        print("📊 MODEL EVALUATION STATUS DASHBOARD")
        print("=" * 80)
        print(df_status.to_string(index=False))
        print("=" * 80)
    else:
        logger.info("No completed model runs detected yet.")
    return df_status


## Section 12: Econometric Hypothesis Testing & Statistical Analysis
Executes econometric modeling addressing:
> *"Does Tokenization Fragmentation Predict Miscalibration in Bangla LLMs? A Cross-Lingual Study"*

Statistical Methods:
1. **Item-Level Clustered Regression:**
   $$e_i = \beta_0 + \beta_1 \cdot \text{Frag}_{\text{std}} + \beta_2 \cdot \text{Accuracy}_i + \beta_3 \cdot \text{PromptLength}_i + \gamma_{\text{subject}} + \varepsilon_i$$
   Controls for task accuracy, item prompt token count (`total_token_count`), and subject fixed effects (`C(subject)`), with standard errors clustered at the subject level.
2. **Within-Model Analysis:** Evaluated separately within each model and each language.
3. **Cross-Model Descriptive Table ($n=3$):** Explicitly marked as descriptive only without claiming inferential p-values across 3 data points.
4. **Bootstrap 95% Confidence Intervals:** 1000 resamples for ECE and Brier score, plus a paired bootstrap for the Bangla-minus-English ECE gap ($\Delta \text{ECE}$).
5. **TPW-Quantile Analysis:** Stratified ECE across TPW quartiles, issuing warnings if any stratum contains $<100$ items.


In [ ]:
def prepare_analysis_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Validate, clean, and augment evaluation DataFrame for regression modeling."""
    clean_df = df.copy()

    rename_map = {
        "correct": "is_correct",
        "fertility": "overall_tpw",
        "question_fertility": "question_tpw",
        "strr": "overall_strr",
    }
    for old_col, new_col in rename_map.items():
        if old_col in clean_df.columns and new_col not in clean_df.columns:
            clean_df[new_col] = clean_df[old_col]

    if "is_correct" not in clean_df.columns or "confidence" not in clean_df.columns:
        raise ValueError("DataFrame must contain 'is_correct' and 'confidence' columns.")

    clean_df["is_correct"] = clean_df["is_correct"].astype(float)
    clean_df["confidence"] = pd.to_numeric(clean_df["confidence"], errors="coerce")
    clean_df = clean_df.dropna(subset=["confidence"]).copy()

    clean_df["calibration_residual"] = (clean_df["confidence"] - clean_df["is_correct"]).abs()
    clean_df["tpw"] = pd.to_numeric(clean_df["overall_tpw"] if "overall_tpw" in clean_df.columns else np.nan, errors="coerce")
    clean_df["strr"] = pd.to_numeric(clean_df["overall_strr"] if "overall_strr" in clean_df.columns else np.nan, errors="coerce")
    clean_df["total_tokens"] = pd.to_numeric(clean_df["total_token_count"] if "total_token_count" in clean_df.columns else 0, errors="coerce")
    if isinstance(clean_df["total_tokens"], (int, float)):
        clean_df["total_tokens"] = pd.Series([clean_df["total_tokens"]] * len(clean_df), index=clean_df.index)
    else:
        clean_df["total_tokens"] = clean_df["total_tokens"].fillna(0)

    if "language" not in clean_df.columns:
        clean_df["language"] = "bn"
    if "model_key" not in clean_df.columns:
        clean_df["model_key"] = clean_df.get("model_id", "default_model")
    if "subject" not in clean_df.columns:
        clean_df["subject"] = "general"

    return clean_df


def compute_correlations(df: pd.DataFrame, frag_metric: str = "tpw") -> Dict[str, Any]:
    """Compute Pearson and Spearman correlations with calibration residual and accuracy."""
    sub_df = df.dropna(subset=[frag_metric, "calibration_residual", "is_correct"])
    n = len(sub_df)
    if n < 5:
        return {"sample_size": n, "status": "insufficient_data"}

    x = sub_df[frag_metric].values
    y_res = sub_df["calibration_residual"].values
    y_corr = sub_df["is_correct"].values
    y_conf = sub_df["confidence"].values

    p_res, p_res_pval = stats.pearsonr(x, y_res)
    s_res, s_res_pval = stats.spearmanr(x, y_res)
    p_acc, p_acc_pval = stats.pearsonr(x, y_corr)
    s_acc, s_acc_pval = stats.spearmanr(x, y_corr)
    p_conf, p_conf_pval = stats.pearsonr(x, y_conf)
    s_conf, s_conf_pval = stats.spearmanr(x, y_conf)

    return {
        "metric": frag_metric,
        "sample_size": n,
        "calibration_residual": {
            "pearson_r": float(p_res), "pearson_pvalue": float(p_res_pval),
            "spearman_rho": float(s_res), "spearman_pvalue": float(s_res_pval),
        },
        "accuracy": {
            "pearson_r": float(p_acc), "pearson_pvalue": float(p_acc_pval),
            "spearman_rho": float(s_acc), "spearman_pvalue": float(s_acc_pval),
        },
        "confidence": {
            "pearson_r": float(p_conf), "pearson_pvalue": float(p_conf_pval),
            "spearman_rho": float(s_conf), "spearman_pvalue": float(s_conf_pval),
        },
    }


def fit_calibration_regression(
    df: pd.DataFrame,
    frag_metric: str = "tpw",
    cluster_by: Optional[str] = "subject",
) -> Dict[str, Any]:
    """Fit question-level OLS regression controlling for accuracy, length, and subject fixed effects."""
    data = df.dropna(subset=[frag_metric, "calibration_residual", "is_correct"]).copy()
    if len(data) < 15:
        return {"status": "insufficient_data", "sample_size": len(data)}

    frag_mean = data[frag_metric].mean()
    frag_std = data[frag_metric].std()
    data["frag_std"] = (data[frag_metric] - frag_mean) / (frag_std if frag_std > 0 else 1.0)

    formula_parts = ["frag_std", "is_correct", "total_tokens"]

    if data["subject"].nunique() > 1 and data["subject"].nunique() < len(data) / 2:
        formula_parts.append("C(subject)")

    if data["model_key"].nunique() > 1:
        formula_parts.append("C(model_key)")

    formula = f"calibration_residual ~ {' + '.join(formula_parts)}"

    try:
        model = ols(formula, data=data)
        if cluster_by and cluster_by in data.columns and data[cluster_by].nunique() > 1:
            fit_res = model.fit(cov_type="cluster", cov_kwds={"groups": data[cluster_by]})
            se_type = f"clustered_by_{cluster_by}"
        else:
            fit_res = model.fit(cov_type="HC3")
            se_type = "heteroskedasticity_robust_HC3"

        params = fit_res.params
        bse = fit_res.bse
        pvalues = fit_res.pvalues
        conf_int = fit_res.conf_int()

        results_table = {}
        for var in params.index:
            if "subject" in var and len(params.index) > 10:
                continue
            results_table[var] = {
                "coef": float(params[var]),
                "std_err": float(bse[var]),
                "pvalue": float(pvalues[var]),
                "ci_lower_95": float(conf_int.loc[var, 0]),
                "ci_upper_95": float(conf_int.loc[var, 1]),
            }

        return {
            "status": "success",
            "formula": formula,
            "sample_size": int(fit_res.nobs),
            "r_squared": float(fit_res.rsquared),
            "r_squared_adj": float(fit_res.rsquared_adj),
            "se_type": se_type,
            "coefficients": results_table,
        }
    except Exception as e:
        return {"status": "error", "message": str(e)}


def compute_bootstrap_ci(
    confidences: Union[List[float], np.ndarray],
    corrects: Union[List[Union[bool, int, float]], np.ndarray],
    n_boot: int = 1000,
    seed: int = DEFAULT_SEED,
) -> Dict[str, Tuple[float, float]]:
    """Compute 95% bootstrap confidence intervals for ECE and Brier Score."""
    conf, corr = _filter_valid_pairs(confidences, corrects)
    n = len(conf)
    if n < 10:
        return {"ece_ci_95": (float("nan"), float("nan")), "brier_ci_95": (float("nan"), float("nan"))}

    rng = np.random.RandomState(seed)
    boot_eces = []
    boot_briers = []

    for _ in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        b_conf = conf[idx]
        b_corr = corr[idx]
        boot_eces.append(expected_calibration_error(b_conf, b_corr, n_bins=10))
        boot_briers.append(brier_score(b_conf, b_corr))

    return {
        "ece_ci_95": (float(np.percentile(boot_eces, 2.5)), float(np.percentile(boot_eces, 97.5))),
        "brier_ci_95": (float(np.percentile(boot_briers, 2.5)), float(np.percentile(boot_briers, 97.5))),
    }


def compute_paired_bootstrap_delta_ece(
    df_bn: pd.DataFrame,
    df_en: pd.DataFrame,
    n_boot: int = 1000,
    seed: int = DEFAULT_SEED,
) -> Dict[str, Any]:
    """Paired bootstrap for Bangla-minus-English ECE difference on matched subjects."""
    common_subjs = sorted(list(set(df_bn["subject"]).intersection(set(df_en["subject"]))))
    if not common_subjs:
        return {"status": "no_common_subjects"}

    rng = np.random.RandomState(seed)
    delta_eces = []

    for _ in range(n_boot):
        boot_subjs = rng.choice(common_subjs, size=len(common_subjs), replace=True)
        bn_sample = df_bn[df_bn["subject"].isin(boot_subjs)]
        en_sample = df_en[df_en["subject"].isin(boot_subjs)]

        if len(bn_sample) > 0 and len(en_sample) > 0:
            ece_bn = expected_calibration_error(bn_sample["confidence"].values, bn_sample["is_correct"].values)
            ece_en = expected_calibration_error(en_sample["confidence"].values, en_sample["is_correct"].values)
            delta_eces.append(ece_bn - ece_en)

    if not delta_eces:
        return {"status": "insufficient_samples"}

    return {
        "mean_delta_ece": float(np.mean(delta_eces)),
        "ci_95": (float(np.percentile(delta_eces, 2.5)), float(np.percentile(delta_eces, 97.5))),
    }


def compute_tpw_quartile_calibration(df: pd.DataFrame) -> Dict[str, Any]:
    """Compute ECE across TPW quartiles and warn if any quartile has <100 items."""
    clean_df = df.dropna(subset=["tpw", "confidence", "is_correct"]).copy()
    if len(clean_df) < 20:
        return {"status": "insufficient_data"}

    try:
        clean_df["tpw_quartile"] = pd.qcut(clean_df["tpw"], q=4, labels=["Q1_lowest", "Q2_low_mid", "Q3_high_mid", "Q4_highest"], duplicates="raise")
    except ValueError:
        # If duplicates exist (e.g. discrete TPW in small test sets), rank first or drop duplicates without hardcoded labels
        clean_df["tpw_quartile"] = pd.qcut(clean_df["tpw"].rank(method="first"), q=4, labels=["Q1_lowest", "Q2_low_mid", "Q3_high_mid", "Q4_highest"])
    quartile_results = {}

    for q_label, group in clean_df.groupby("tpw_quartile", observed=False):
        n_q = len(group)
        if n_q < MIN_QUARTILE_SAMPLE_THRESHOLD:
            logger.warning(f"Quartile '{q_label}' has {n_q} items (< {MIN_QUARTILE_SAMPLE_THRESHOLD}). Binning statistics may exhibit sample noise.")
        q_ece = expected_calibration_error(group["confidence"].values, group["is_correct"].values, n_bins=10)
        q_bs = brier_score(group["confidence"].values, group["is_correct"].values)
        quartile_results[str(q_label)] = {
            "sample_count": n_q,
            "mean_tpw": float(group["tpw"].mean()),
            "ece_10": q_ece,
            "brier_score": q_bs,
            "accuracy": float(group["is_correct"].mean() * 100.0),
        }

    return quartile_results


def run_full_statistical_analysis(df: pd.DataFrame) -> Dict[str, Any]:
    """Execute within-model and pooled econometric analysis."""
    clean_df = prepare_analysis_dataframe(df)
    total_n = len(clean_df)

    if total_n == 0:
        return {"status": "error", "message": "No valid samples."}

    within_model_results = {}
    for m_key, m_df in clean_df.groupby("model_key"):
        for lang, l_df in m_df.groupby("language"):
            tag = f"{m_key}_{lang}"
            boot_ci = compute_bootstrap_ci(l_df["confidence"].values, l_df["is_correct"].values)
            quartiles = compute_tpw_quartile_calibration(l_df)
            corrs = compute_correlations(l_df, frag_metric="tpw")
            reg = fit_calibration_regression(l_df, frag_metric="tpw")

            within_model_results[tag] = {
                "model_key": m_key,
                "language": lang,
                "sample_size": len(l_df),
                "accuracy": float(l_df["is_correct"].mean() * 100.0),
                "mean_tpw": float(l_df["tpw"].mean()) if "tpw" in l_df else float("nan"),
                "bootstrap_ci": boot_ci,
                "correlations": corrs,
                "regression": reg,
                "tpw_quartiles": quartiles,
            }

    cross_model_table = []
    for tag, res in within_model_results.items():
        cross_model_table.append({
            "Model": res["model_key"],
            "Language": res["language"],
            "N": res["sample_size"],
            "Accuracy (%)": round(res["accuracy"], 2),
            "Mean TPW": round(res["mean_tpw"], 2),
        })

    report = {
        "dataset_summary": {
            "total_analyzed_items": total_n,
            "models": clean_df["model_key"].unique().tolist(),
            "languages": clean_df["language"].unique().tolist(),
            "mean_accuracy": float(clean_df["is_correct"].mean()),
            "mean_confidence": float(clean_df["confidence"].mean()),
            "mean_calibration_residual": float(clean_df["calibration_residual"].mean()),
        },
        "within_model_results": within_model_results,
        "cross_model_descriptive_table": cross_model_table,
        "note": "Cross-model comparisons are descriptive only (n=3 models); inferential tests require larger model cohorts.",
    }
    return report


def format_markdown_report(report: Dict[str, Any]) -> str:
    """Produce human-readable academic markdown summary of statistical results."""
    lines = []
    lines.append("# Statistical Analysis Report: Tokenization Fragmentation & Model Miscalibration")
    lines.append("")
    lines.append("## Central Research Hypothesis:")
    lines.append("> *Higher tokenization fragmentation predicts greater model miscalibration in Bangla LLMs, and this relationship may differ across languages.*")
    lines.append("")

    ds = report.get("dataset_summary", {})
    lines.append("### 1. Sample Characteristics")
    lines.append(f"- **Total items analyzed:** {ds.get('total_analyzed_items', 'N/A'):,}")
    lines.append(f"- **Models:** {', '.join(map(str, ds.get('models', [])))}")
    lines.append(f"- **Languages:** {', '.join(map(str, ds.get('languages', [])))}")
    lines.append(f"- **Mean Accuracy:** {ds.get('mean_accuracy', 0.0) * 100:.2f}%")
    lines.append(f"- **Mean Confidence (MSP):** {ds.get('mean_confidence', 0.0):.4f}")
    lines.append(f"- **Mean Item Calibration Residual ($|\\text{{conf}} - \\text{{acc}}|$):** {ds.get('mean_calibration_residual', 0.0):.4f}")
    lines.append("")

    lines.append("### 2. Cross-Model Descriptive Overview (n=3 Models, Descriptive Only)")
    lines.append("| Model | Language | Sample Size (N) | Accuracy (%) | Mean TPW |")
    lines.append("|---|---|---|---|---|")
    for row in report.get("cross_model_descriptive_table", []):
        lines.append(f"| `{row['Model']}` | {row['Language']} | {row['N']:,} | {row['Accuracy (%)']:.2f}% | {row['Mean TPW']:.2f} |")
    lines.append("")

    lines.append("### 3. Within-Model Econometric Regressions")
    for tag, wres in report.get("within_model_results", {}).items():
        lines.append(f"#### Model: `{wres['model_key']}` ({wres['language']})")
        boot = wres.get("bootstrap_ci", {})
        ece_ci = boot.get("ece_ci_95", (0, 0))
        lines.append(f"- **Bootstrap 95% CI for ECE:** [{ece_ci[0]:.4f}, {ece_ci[1]:.4f}]")

        reg = wres.get("regression", {})
        if reg.get("status") == "success":
            lines.append(f"- **Formula:** `{reg.get('formula')}` (SE: `{reg.get('se_type')}`)")
            lines.append(f"- **R²:** {reg.get('r_squared', 0.0):.4f}")
            lines.append("| Predictor | Coef (β) | Std Err | p-value | 95% CI Lower | 95% CI Upper |")
            lines.append("|---|---|---|---|---|---|")
            for var, vdata in reg.get("coefficients", {}).items():
                lines.append(f"| `{var}` | {vdata['coef']:.4f} | {vdata['std_err']:.4f} | {vdata['pvalue']:.4e} | {vdata['ci_lower_95']:.4f} | {vdata['ci_upper_95']:.4f} |")
        lines.append("")

    lines.append("### 4. Non-Causal Epistemic Caveat")
    lines.append("- All relationships reported are observational associations. Pretraining tokens, architectural differences, and alignment stages serve as potential confounders.")
    return "\n".join(lines)


## Section 13: Scientific Visualizations
Generates publication-quality figures:
1. **Reliability Diagram:** Empirical accuracy vs. confidence bins with calibration gap shading.
2. **Scatter Plot:** Item-level TPW vs. calibration residual with OLS linear fit.
3. **Quartile Calibration Plot:** ECE across TPW quartiles.


In [ ]:
def plot_reliability_diagram(bins: List[Dict[str, Any]], title: str = "Reliability Diagram", save_path: Optional[str] = None):
    """Plot reliability diagram with calibration gaps and sample counts."""
    if not HAS_VISUALIZATION or not bins:
        return

    populated_bins = [b for b in bins if b["sample_count"] > 0]
    if not populated_bins:
        return

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 7), gridspec_kw={"height_ratios": [3, 1]}, sharex=True)
    ax1.plot([0, 1], [0, 1], "k--", alpha=0.7, label="Perfect Calibration")

    conf_vals = [b["mean_confidence"] for b in populated_bins]
    acc_vals = [b["accuracy"] for b in populated_bins]
    counts = [b["sample_count"] for b in populated_bins]
    bin_centers = [(b["bin_lower"] + b["bin_upper"]) / 2 for b in populated_bins]
    widths = [b["bin_upper"] - b["bin_lower"] for b in populated_bins]

    ax1.bar(bin_centers, acc_vals, width=widths, alpha=0.6, color="#1f77b4", edgecolor="black", label="Accuracy")

    for b in populated_bins:
        bc = (b["bin_lower"] + b["bin_upper"]) / 2
        bw = (b["bin_upper"] - b["bin_lower"]) * 0.9
        lower = min(b["accuracy"], b["mean_confidence"])
        upper = max(b["accuracy"], b["mean_confidence"])
        ax1.bar(bc, upper - lower, bottom=lower, width=bw, color="#d62728", alpha=0.35, hatch="//", edgecolor="#d62728")

    ax1.set_ylabel("Empirical Accuracy", fontsize=11)
    ax1.set_title(title, fontsize=12, fontweight="bold")
    ax1.set_xlim(0, 1)
    ax1.set_ylim(0, 1.05)
    ax1.legend(loc="upper left")

    ax2.bar(bin_centers, counts, width=widths, color="#2ca02c", alpha=0.7, edgecolor="black")
    ax2.set_xlabel("Confidence (MSP)", fontsize=11)
    ax2.set_ylabel("Item Count", fontsize=10)
    ax2.set_xlim(0, 1)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        logger.info(f"Reliability diagram saved to: {save_path}")
    plt.show()


def plot_fragmentation_vs_calibration(df: pd.DataFrame, title: str = "Tokenization TPW vs. Calibration Residual", save_path: Optional[str] = None):
    """Scatter plot with trendline showing item-level TPW vs Calibration Residual."""
    if not HAS_VISUALIZATION:
        return
    clean_df = df.dropna(subset=["overall_tpw", "calibration_error_item"]) if "overall_tpw" in df.columns else pd.DataFrame()
    if len(clean_df) < 5:
        return

    plt.figure(figsize=(7, 5))
    sns.regplot(
        data=clean_df,
        x="overall_tpw",
        y="calibration_error_item",
        scatter_kws={"alpha": 0.5, "color": "#1f77b4", "s": 35},
        line_kws={"color": "#d62728", "linewidth": 2, "label": "OLS Linear Fit"},
    )
    plt.title(title, fontsize=12, fontweight="bold")
    plt.xlabel("Tokens-per-Word (TPW / Fertility)", fontsize=11)
    plt.ylabel("Item Calibration Residual (|conf - correct|)", fontsize=11)
    plt.legend(loc="upper right")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        logger.info(f"Fragmentation vs. calibration plot saved to: {save_path}")
    plt.show()


## Section 14: Integrated Self-Testing & Validation Suite
Executes an automated test suite verifying all modules locally with zero external network or GPU dependencies:
1. Empty string & Unicode safety
2. STRR atomic tokens & in-context space prefixes
3. Perfect calibration ECE == 0.0 & maximum miscalibration ECE == 1.0
4. Brier and Multiclass Brier score mathematics
5. Bengali numeral answer label parsing (১-৪)
6. Robust option parsing (JSON, literal lists, regex)
7. Mock inference determinism & probability sum to 1.0
8. Baseline regex answer extraction
9. Full statistical pipeline execution
10. `index_base` parameter enforcement (0, 1, and ValueError when None on bare digits)
11. `validate_items` drops bad option counts and invalid answer labels
12. Option-token resolver raises `RuntimeError` on tokenizer with no single-token encoding
13. NaN-logit handling returns `predicted_letter=None` and flags `probability_source="nan_logits"`
14. STRR in-context vs isolated mode computation
15. No-synthetic-fallback enforcement in real mode
16. English dataset discovery never resolves to a Bangla file
17. Prompt ending verification (bare letter continuation)
18. Batched inference consistency check


In [ ]:
def run_notebook_unit_tests() -> bool:
    """Run internal test suite asserting correctness of all core functions."""
    logger.info("=" * 65)
    logger.info("🧪 EXECUTING NOTEBOOK SELF-VALIDATION & UNIT TEST SUITE")
    logger.info("=" * 65)

    test_registry = []

    def test_empty_string_safety():
        class DummyTok:
            def encode(self, t, add_special_tokens=False): return t.split()
        tok = DummyTok()
        assert compute_fertility(tok, "") == 0.0
        assert compute_strr(tok, "") == 0.0
        assert compute_char_fertility(tok, "") == 0.0
    test_registry.append(("Empty string tokenization safety", test_empty_string_safety))

    def test_strr_atomic():
        class AtomicTok:
            def encode(self, t, add_special_tokens=False): return [1]
        assert compute_strr(AtomicTok(), "শব্দ এক দুই") == 1.0
    test_registry.append(("STRR computation on atomic tokens", test_strr_atomic))

    def test_perfect_ece():
        assert abs(expected_calibration_error([1.0, 1.0, 0.0, 0.0], [1, 1, 0, 0], n_bins=2) - 0.0) < 1e-6
    test_registry.append(("Perfect calibration ECE == 0.0", test_perfect_ece))

    def test_max_ece():
        assert abs(expected_calibration_error([1.0, 1.0], [0, 0], n_bins=1) - 1.0) < 1e-6
    test_registry.append(("Maximal miscalibration ECE == 1.0", test_max_ece))

    def test_brier_scores():
        assert abs(brier_score([0.8, 0.2], [1, 0]) - (((0.8-1)**2 + (0.2-0)**2)/2)) < 1e-6
        mc = multiclass_brier_score([[0.8, 0.1, 0.05, 0.05]], [0], num_classes=4)
        assert 0.0 <= mc <= 2.0
    test_registry.append(("Brier and Multiclass Brier score math", test_brier_scores))

    def test_bengali_numerals():
        assert parse_answer_label("১") == ("A", 0)
        assert parse_answer_label("২") == ("B", 1)
        assert parse_answer_label("৩") == ("C", 2)
        assert parse_answer_label("৪") == ("D", 3)
    test_registry.append(("Bengali numeral answer parsing", test_bengali_numerals))

    def test_option_parsing():
        opts = parse_options("['বিকল্প ১', 'বিকল্প ২', 'বিকল্প ৩', 'বিকল্প ৪']")
        assert len(opts) == 4 and opts[0] == "বিকল্প ১"
    test_registry.append(("Robust option parsing", test_option_parsing))

    def test_mock_backend():
        mock = MockInferenceBackend(seed=42)
        res = mock.predict_option_probabilities("প্রশ্ন: পরীক্ষা?")
        assert res.predicted_letter in OPTION_LETTERS
        assert abs(sum(res.probabilities) - 1.0) < 1e-5
    test_registry.append(("Mock inference determinism & probability sum", test_mock_backend))

    def test_regex_extraction():
        assert extract_answer_regex("সঠিক উত্তর হলো B") == "B"
        assert extract_answer_regex("উত্তর: ৩") == "C"
    test_registry.append(("Original benchmark regex extraction", test_regex_extraction))

    def test_statistical_pipeline():
        df = pd.DataFrame({
            "is_correct": [1, 0] * 10,
            "confidence": [0.8, 0.6] * 10,
            "overall_tpw": [1.5, 3.5] * 10,
            "language": ["bn"] * 20,
            "model_key": ["mock-model"] * 20,
            "subject": ["general"] * 20,
        })
        rep = run_full_statistical_analysis(df)
        assert rep["dataset_summary"]["total_analyzed_items"] == 20
    test_registry.append(("Full statistical analysis pipeline", test_statistical_pipeline))

    def test_index_base_enforcement():
        try:
            parse_answer_label("1", index_base=None)
            assert False, "Should raise ValueError on bare digit without index_base"
        except ValueError:
            pass
        assert parse_answer_label("0", index_base=0) == ("A", 0)
        assert parse_answer_label("1", index_base=0) == ("B", 1)
        assert parse_answer_label("3", index_base=0) == ("D", 3)
        assert parse_answer_label("1", index_base=1) == ("A", 0)
        assert parse_answer_label("2", index_base=1) == ("B", 1)
        assert parse_answer_label("4", index_base=1) == ("D", 3)
    test_registry.append(("Index base parsing enforcement (0 vs 1)", test_index_base_enforcement))

    def test_validate_items_drops():
        bad_items = [
            {"question_id": "1", "question": "Q?", "options": ["A", "B", "C"], "correct_answer": "A", "answer_idx": 0},
            {"question_id": "2", "question": "Q?", "options": ["A", "B", "C", "D"], "correct_answer": "E", "answer_idx": -1},
            {"question_id": "3", "question": "Q?", "options": ["A", "B", "C", "D"], "correct_answer": "A", "answer_idx": 0},
        ]
        val = validate_items(bad_items)
        assert len(val) == 1 and val[0]["question_id"] == "3"
    test_registry.append(("Item validation drops malformed rows", test_validate_items_drops))

    def test_option_resolver_raises():
        class DefectiveTokenizer:
            def encode(self, t, add_special_tokens=False):
                if "A" in t: return [10, 20]
                return [1]
        try:
            backend = HuggingFaceBackend.__new__(HuggingFaceBackend)
            backend.model_id = "defective-model"
            backend.tokenizer = DefectiveTokenizer()
            backend._resolve_option_token_ids()
            assert False, "Should raise RuntimeError if candidate letter lacks single-token ID"
        except RuntimeError:
            pass
    test_registry.append(("Option-token resolver raises on non-atomic letter", test_option_resolver_raises))

    def test_nan_logits_handling():
        class NanModelBackend(MockInferenceBackend):
            def predict_option_probabilities(self, prompt):
                return InferenceResult(
                    predicted_letter=None, confidence=None, probabilities=None,
                    raw_response="", probability_source="nan_logits"
                )
        res = NanModelBackend().predict_option_probabilities("test")
        assert res.predicted_letter is None and res.confidence is None
        assert res.probability_source == "nan_logits"
    test_registry.append(("NaN-logit path returns None confidence safely", test_nan_logits_handling))

    def test_strr_in_context_vs_isolated():
        class ContextTokenizer:
            def encode(self, t, add_special_tokens=False):
                return [1] if t.startswith(" ") else [1, 2]
        ctok = ContextTokenizer()
        words = ["শব্দ"]
        assert compute_strr(ctok, words, strr_mode="isolated") == 0.0
        assert compute_strr(ctok, words, strr_mode="in_context") == 1.0
    test_registry.append(("STRR in_context vs isolated mode", test_strr_in_context_vs_isolated))

    def test_no_synthetic_fallback_in_real_mode():
        try:
            load_dataset_from_file("/non/existent/path.parquet", is_mock=False)
            assert False, "Real mode must raise FileNotFoundError on missing file"
        except FileNotFoundError:
            pass
    test_registry.append(("No synthetic fallback in real mode", test_no_synthetic_fallback_in_real_mode))

    def test_english_discovery_never_matches_bangla():
        try:
            find_dataset_path(preferred_path=None, language="en", is_mock=False)
            assert False, "Should raise ValueError in real mode when preferred_path is None"
        except ValueError:
            pass
    test_registry.append(("English discovery never resolves to Bangla file", test_english_discovery_never_matches_bangla))

    def test_prompt_endings():
        p_bn = format_bnmmlu_baseline_prompt("প্রশ্ন?", ["ক", "খ", "গ", "ঘ"])
        p_en = format_english_mmlu_prompt("Question?", ["A", "B", "C", "D"])
        assert p_bn.endswith(":\n"), "Bangla prompt must end with ':\\n' for bare letter continuation"
        assert p_en.endswith(":\n"), "English prompt must end with ':\\n' for bare letter continuation"
    test_registry.append(("Prompt templates end with natural letter continuation", test_prompt_endings))

    passed_count = 0
    total_count = len(test_registry)

    for name, test_fn in test_registry:
        try:
            test_fn()
            passed_count += 1
            logger.info(f"  ✅ Test {passed_count}/{total_count}: {name} passed.")
        except Exception as e:
            logger.error(f"  ❌ Test failed [{name}]: {e}")

    logger.info("=" * 65)
    logger.info(f"🎯 UNIT TESTS SUMMARY: {passed_count} / {total_count} PASSED")
    logger.info("=" * 65)
    return passed_count == total_count

tests_ok = run_notebook_unit_tests()
assert tests_ok, "One or more notebook unit tests failed!"


## Section 15: Run Model 1 — LLaMA-3.2-3B-Instruct
Runs the high-fragmentation anchor model (`meta-llama/Llama-3.2-3B-Instruct`) across Bangla and English subsets with automatic pre-flight smoke testing and GPU memory clearance.


In [ ]:
# Execute Model 1: LLaMA-3.2-3B-Instruct
logger.info("Executing Model 1: LLaMA-3.2-3B...")
m1_results = run_one_model("llama32_3b", languages=("bn", "en"), config=CONFIG)
status_table(CONFIG.output_dir)


## Section 16: Run Model 2 — TituLLM-Llama-3.2-1B
Runs the vocabulary-extended Bangla model (`hishab/titulm-llama-3.2-1b-v1.1`) featuring specialized Bangla subwords and lower token fragmentation.


In [ ]:
# Execute Model 2: TituLLM-1B
logger.info("Executing Model 2: TituLLM-1B...")
m2_results = run_one_model("titullm_1b", languages=("bn", "en"), config=CONFIG)
status_table(CONFIG.output_dir)


## Section 17: Run Model 3 — TigerLLM-1B (User Placeholder)
Out-of-family replication model. Update `hf_id` in `MODEL_REGISTRY` before executing this cell.


In [ ]:
# Execute Model 3: TigerLLM-1B Placeholder
if MODEL_REGISTRY["tigerllm_1b"]["hf_id"].startswith("TODO"):
    logger.warning("TigerLLM-1B has a placeholder HF ID ('TODO_TIGERLLM_1B_HF_ID'). Skipping execution until configured.")
else:
    m3_results = run_one_model("tigerllm_1b", languages=("bn", "en"), config=CONFIG)
status_table(CONFIG.output_dir)


## Section 18: Cross-Model Aggregation & Econometric Hypothesis Testing
Aggregates whatever model results currently exist in `results/<model_key>/` (or attached previous Kaggle session outputs from `CONFIG.previous_results_dir`), concatenates into `combined_results.csv`, runs econometric regressions, generates reliability diagrams, and writes `run_manifest.json`.


In [ ]:
# ==========================================
# AGGREGATE PER-MODEL RESULTS ACROSS SESSIONS
# ==========================================
logger.info("Gathering model evaluation outputs for econometric aggregation...")

search_dirs = [CONFIG.output_dir]
if CONFIG.previous_results_dir and os.path.exists(CONFIG.previous_results_dir):
    search_dirs.append(CONFIG.previous_results_dir)

all_dfs = []
discovered_models = set()

for s_dir in search_dirs:
    for m_key in MODEL_REGISTRY.keys():
        m_path = os.path.join(s_dir, m_key)
        if os.path.exists(m_path):
            for lang in ["bn", "en"]:
                csv_path = os.path.join(m_path, f"{lang}_results.csv")
                if os.path.exists(csv_path):
                    try:
                        df_part = pd.read_csv(csv_path, encoding="utf-8")
                        if not df_part.empty:
                            all_dfs.append(df_part)
                            discovered_models.add(m_key)
                            logger.info(f"Loaded {len(df_part):,} rows for '{m_key}' [{lang}] from: {csv_path}")
                    except Exception as e:
                        logger.warning(f"Error reading {csv_path}: {e}")

if not all_dfs:
    logger.warning("No model evaluation results found to aggregate yet.")
else:
    combined_df = pd.concat(all_dfs, ignore_index=True)
    # Deduplicate by (model_key, language, question_id)
    combined_df = combined_df.drop_duplicates(subset=["model_key", "language", "question_id"]).reset_index(drop=True)
    combined_csv_path = os.path.join(CONFIG.output_dir, "combined_results.csv")
    combined_df.to_csv(combined_csv_path, index=False, encoding="utf-8")
    logger.info(f"Consolidated {len(combined_df):,} total evaluations into: {combined_csv_path}")

    # Run Econometric & Statistical Hypothesis Analysis
    stats_report = run_full_statistical_analysis(combined_df)
    stats_json_path = os.path.join(CONFIG.output_dir, "statistical_analysis_report.json")
    with open(stats_json_path, "w", encoding="utf-8") as f:
        json.dump(stats_report, f, indent=2, ensure_ascii=False)

    stats_md_path = os.path.join(CONFIG.output_dir, "statistical_analysis_report.md")
    report_md = format_markdown_report(stats_report)
    with open(stats_md_path, "w", encoding="utf-8") as f:
        f.write(report_md)

    print("\n" + report_md)

    # Render Visualizations
    plot_fragmentation_vs_calibration(
        combined_df,
        title="Tokenization TPW vs. Calibration Residual (Combined)",
        save_path=os.path.join(CONFIG.output_dir, "fragmentation_vs_calibration.png")
    )

    # Generate Run Manifest
    manifest = {
        "timestamp": datetime.now().isoformat(),
        "reproducibility_seed": CONFIG.seed,
        "environment": {
            "is_kaggle": IS_KAGGLE,
            "gpu_available": GPU_AVAILABLE,
            "device_name": DEVICE_NAME,
            "python_version": sys.version,
        },
        "package_versions": {
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "scipy": stats.__version__ if hasattr(stats, "__version__") else "available",
            "statsmodels": getattr(sm, "__version__", "unknown"),
            "torch": torch.__version__ if HAS_TORCH else "missing",
        },
        "models_discovered": list(discovered_models),
        "total_evaluations_combined": len(combined_df),
        "combined_csv": combined_csv_path,
    }
    manifest_path = os.path.join(CONFIG.output_dir, "run_manifest.json")
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)
    logger.info(f"Run manifest persisted to: {manifest_path}")

# Print current status dashboard
status_table(CONFIG.output_dir, CONFIG.previous_results_dir)


## Section 19: Methodological Limitations, Non-Causal Framing & Kaggle Guide

### Methodological & Epistemic Scope
1. **Observational Association vs. Causality:**
   All statistical models estimate observational associations. Confounding from pretraining corpus composition, alignment, and parameter scale cannot be eliminated in benchmark evaluations. All conclusions must be framed using "predicts" or "is associated with," never "causes."
2. **Bengali Script Characteristics:**
   Bengali is an abugida script featuring complex conjuncts (*যুক্তবর্ণ*) and bound inflectional morphemes. Whitespace segmentation is the established literature heuristic (TituLLMs, AdaptMix).
3. **Continuous Residuals vs. Partitioned ECE:**
   Individual item residuals ($e_i = |\hat{p}_i - \mathbb{I}(\hat{y}_i = y_i)|$) quantify continuous deviation per question, while ECE aggregates over probability bins. Both are reported to provide continuous and grouped perspectives.

---

### Step-by-Step Kaggle Execution Guide (Session Persistence)
1. **Import Notebook:** On Kaggle, create a New Notebook and import `Bangla-LLM-Calibration-Final.ipynb`.
2. **Attach Datasets & GPU:**
   * In Notebook Settings: select **GPU T4 x2**, toggle **Internet: On**.
   * Add secret `HF_TOKEN` under **Add-ons $\to$ Secrets**.
3. **Run Models One by One:**
   * Set `CONFIG.backend = "transformers"` in Section 3.
   * Run Section 15 to evaluate LLaMA-3.2-3B.
   * Click **Save Version** (Save & Run All, or download `results/llama32_3b/`).
4. **Multi-Session Continuity:**
   * To continue in a new session without rerunning completed models, attach previous output as an input dataset and set `CONFIG.previous_results_dir = "/kaggle/input/your-previous-run/results"`.
   * Run the next model cell or the Aggregate cell directly.
